# Cella 1 - Installazione dipendenze

In [ ]:
!apt-get install -q -y ffmpeg
!pip install -q rapidfuzz sentence-transformers
!pip install -q transformers>=4.40.0 accelerate librosa
!pip install json-repair

!pip install -q hydra-core lightning panphon phonemizer
!pip install -q torchaudio huggingface_hub

!pip install -q spacy
!python -m spacy download it_core_news_lg -q
!pip install pymongo

!pip install -q google-api-python-client

!pip install -q sacrebleu rouge-score jiwer

from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from jiwer import wer

from googleapiclient.discovery import build
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import uuid
from datetime import datetime, timezone

from pathlib import Path
import json as _json

import traceback


from transformers import BertTokenizerFast, BertForMaskedLM
import math



import os
import sys
import time
import string
import torch
import numpy as np
import pandas as pd
import librosa
import json
import gc
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline,
)

import copy

import re
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer

import torchaudio
from huggingface_hub import hf_hub_download
from openai import OpenAI
import json_repair
import spacy

from openai import OpenAI

print('Dipendenze installate')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.

# Cella 2.1 - Verifica GPU

In [ ]:

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Nessuna GPU — Whisper large sara lento, considera whisper-base')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


# Cella 2.2 - Download Modelli

In [ ]:
# Processore per model (large-v3 — 128 canali mel)
processor = AutoProcessor.from_pretrained("openai/whisper-large-v3")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-large-v3",
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(DEVICE).eval()


!git clone https://github.com/changelinglab/PhoneticXeus.git
sys.path.append("/kaggle/working/PhoneticXeus")

from src.model.xeusphoneme.builders import build_xeus_pr_inference

REPO = "changelinglab/PhoneticXeus"
ckpt_path = hf_hub_download(REPO, "phoneticxeus_state_dict.pt")
# Il file vocab in realtà si trova già nella cartella scaricata da GitHub
vocab_path = "/kaggle/working/PhoneticXeus/src/model/xeusphoneme/resources/ipa_vocab.json"

    # Costruiamo l'inferenza rimuovendo ctc_weight e altri argomenti non supportati
inference = build_xeus_pr_inference(
work_dir="exp/cache/xeus",
hf_repo="espnet/xeus",
checkpoint=ckpt_path,
vocab_file=vocab_path,
device=DEVICE,
interctc_use_conditioning=True, # Questo è l'unico parametro extra supportato qui
)

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

Cloning into 'PhoneticXeus'...
remote: Enumerating objects: 429, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (326/326), done.
remote: Total 429 (delta 131), reused 383 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (429/429), 380.41 KiB | 9.51 MiB/s, done.
Resolving deltas: 100% (131/131), done.


phoneticxeus_state_dict.pt:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loaded checkpoint: /root/.cache/huggingface/hub/models--changelinglab--PhoneticXeus/snapshots/bf28fd7958b9c20a268f7e93ce62ee1748713889/phoneticxeus_state_dict.pt with load info: <All keys matched successfully>


In [ ]:
import time
from openai import OpenAI

# ── Lista di API KEY NVIDIA (aggiungine quante vuoi) ──────────
NVIDIA_API_KEYS = [
    "nvapi-key-1",
    "nvapi-key-2",
    "nvapi-key-3",
    "...",
]

_current_key_index = 0

def _get_nvidia_client() -> OpenAI:
    """Restituisce un client NVIDIA con la chiave attiva."""
    return OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEYS[_current_key_index],
    )

def _rotate_key():
    """Ruota alla chiave successiva. Solleva eccezione se le ha esaurite tutte."""
    global _current_key_index
    _current_key_index += 1
    if _current_key_index >= len(NVIDIA_API_KEYS):
        _current_key_index = 0  # oppure: raise RuntimeError("Tutte le API KEY esaurite")
        raise RuntimeError("Tutte le API KEY NVIDIA hanno ricevuto 429 — riprova più tardi.")
    print(f"  🔄 Rotazione chiave NVIDIA → indice {_current_key_index}")

MODEL = "mistralai/mistral-large-3-675b-instruct-2512"

# Cella 2.3 - Caricamento Modello BERT Italiano (PPPL)

In [ ]:
# ══════════════════════════════════════════════════════════════
# PPPL — Pseudo-Perplexity su testo italiano normalizzato
# Modello: dbmdz/bert-base-italian-cased
# Motivazione: MLM monolingue italiano, cased per termini medici
# Formula: PPPL = exp( -1/N * sum_i log P(w_i | w_{\i}) )
# ══════════════════════════════════════════════════════════════

PPPL_MODEL_NAME = "dbmdz/bert-base-italian-cased"

print(f"Caricamento BERT italiano per PPPL: {PPPL_MODEL_NAME}")
_pppl_tokenizer = BertTokenizerFast.from_pretrained(PPPL_MODEL_NAME)
_pppl_model     = BertForMaskedLM.from_pretrained(PPPL_MODEL_NAME)
_pppl_model.eval()

# Sposta su GPU se disponibile — il modello è leggero (110M params)
_pppl_model = _pppl_model.to(DEVICE)
print(f"✅ Modello PPPL caricato su {DEVICE}")

def _normalize_pppl(pppl_value: float,
                    n_tokens: int = None,
                    pppl_min: float = 5.0,
                    pppl_max: float = 500.0) -> float:
    if pppl_value is None:
        return 0.5
    # Testo troppo corto: PPPL non ha significato statistico, valore neutro
    if n_tokens is not None and n_tokens < 3:
        return 0.1
    normed = (pppl_value - pppl_min) / (pppl_max - pppl_min)
    return float(np.clip(normed, 0.0, 1.0))

def compute_pppl(text: str,
                 tokenizer=_pppl_tokenizer,
                 model=_pppl_model,
                 pppl_min: float = 5.0,
                 pppl_max: float = 500.0,
                 max_length: int = 512) -> dict:
    """
    Calcola la Pseudo-Perplexity (PPPL) su un testo italiano normalizzato.

    Algoritmo:
      Per ogni token i nella sequenza:
        1. Maschera il token i con [MASK]
        2. Ottieni P(w_i | w_{\\i}) dalla testa MLM di BERT
        3. Accumula log P
      PPPL = exp( -1/N * sum log P )

    Interpretazione:
      - PPPL bassa  → testo fluente, italiano standard atteso
      - PPPL alta   → testo insolito, anomalie semantiche o errori ASR residui

    Args:
      text       : testo italiano normalizzato (output di analyze_and_normalize_with_llm)
      tokenizer  : BertTokenizerFast precaricato
      model      : BertForMaskedLM precaricato
      max_length : lunghezza massima token (default 512, limite BERT)

    Returns:
      dict con:
        'pppl'         : float  — valore PPPL (più basso = migliore)
        'log_pppl'     : float  — log2(PPPL) per confronti su scale diverse
        'n_tokens'     : int    — numero di token analizzati
        'mean_log_prob': float  — media dei log P (prima di exp, negato)
        'model'        : str    — nome modello usato
    """
    if not text or not text.strip():
        return {
            'pppl':          None,
            'log_pppl':      None,
            'n_tokens':      0,
            'mean_log_prob': None,
            'model':         PPPL_MODEL_NAME,
        }

    # Tokenizza senza truncation prima per verificare lunghezza
    encodings = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )

    input_ids = encodings['input_ids'].to(DEVICE)  # shape: (1, seq_len)
    seq_len   = input_ids.shape[1]

    # I token speciali [CLS] e [SEP] non vengono mascherati
    # (indici 0 e seq_len-1)
    mask_token_id = tokenizer.mask_token_id
    total_log_prob = 0.0
    n_tokens       = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):  # esclude [CLS] e [SEP]
            # Crea una copia e maschera la posizione i
            masked_ids       = input_ids.clone()
            masked_ids[0, i] = mask_token_id

            outputs = model(masked_ids)
            logits  = outputs.logits  # shape: (1, seq_len, vocab_size)

            # Log-softmax sulla distribuzione al token mascherato
            log_probs    = torch.nn.functional.log_softmax(logits[0, i], dim=-1)
            true_token   = input_ids[0, i].item()
            log_prob_tok = log_probs[true_token].item()

            total_log_prob += log_prob_tok
            n_tokens       += 1

    if n_tokens == 0:
        return {
            'pppl':          None,
            'log_pppl':      None,
            'n_tokens':      0,
            'mean_log_prob': None,
            'model':         PPPL_MODEL_NAME,
        }

    mean_log_prob = total_log_prob / n_tokens          # negativo
    pppl          = round(math.exp(-mean_log_prob), 4) # exp(-(-|x|)) = exp(|x|)
    log_pppl      = round(math.log2(pppl), 4) if pppl > 0 else None

    pppl_norm = _normalize_pppl(pppl, n_tokens=n_tokens, pppl_min=pppl_min, pppl_max=pppl_max)

    return {
        'pppl':          pppl,
        'log_pppl':      log_pppl,
        'n_tokens':      n_tokens,
        'mean_log_prob': round(mean_log_prob, 6),
        'pppl_norm':     pppl_norm,
        'model':         PPPL_MODEL_NAME,
    }


print("✅ Funzione compute_pppl definita")


Caricamento BERT italiano per PPPL: dbmdz/bert-base-italian-cased


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: dbmdz/bert-base-italian-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modello PPPL caricato su cuda
✅ Funzione compute_pppl definita


# Cella 3.1 - Configurazione

In [ ]:
CONFIG = {
    # Embedding multilingue leggero — gira su CPU, non occupa VRAM
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',

    # Quante frasi napoletane nel prompt dinamico
    'num_frasi_prompt': 4,

    # Soglia fuzzy matching (0-100)
    'fuzzy_threshold': 72,

    # Limite caratteri prompt (Whisper: ~224 token ~ 800 chars)
    'max_prompt_chars': 800,
}

print('Config caricata')

Config caricata


# Cella 3.2 - Configurazione MongoDB Atlas

In [ ]:
user_secrets = UserSecretsClient()
MONGO_URI = user_secrets.get_secret("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client['asr_dialects_no_analisi_sintattica']

col_sessions            = db['Sessions']
col_transcripts         = db['Transcripts']
col_risk_logs           = db['Risk_Logs']
col_pipeline_stages     = db['Pipeline_Stages']
col_translation_metrics = db['Translation_Metrics']
col_knowledge           = db['Dialect_KnowledgeBase']
recordings_col          = db['recordings']


print("✅ Connesso a MongoDB Atlas e collezioni inizializzate!")
print(f"   Documenti in Dialect_KnowledgeBase: {col_knowledge.count_documents({})}")

✅ Connesso a MongoDB Atlas e collezioni inizializzate!
   Documenti in Dialect_KnowledgeBase: 0


# Cella 3.3 - Popolamento MongoDB Atlas (Dialect_KnowledgeBase)

In [ ]:
# ══════════════════════════════════════════════════════════════
# POPOLAMENTO MongoDB Atlas — Dialect_KnowledgeBase
# Eseguire UNA SOLA VOLTA (o quando il JSON cambia)
# ══════════════════════════════════════════════════════════════
PERCORSO_JSON   = '/kaggle/input/datasets/matteodavinoaaa/neapolitan-db-m/db_neapolitan.json'
EMBEDDING_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
EMBEDDING_DIM   = 384  # dimensione output del modello scelto

# ── 1. Controlla se la collection è già popolata ──────────────
n_existing = col_knowledge.count_documents({"napoletano": {"$exists": True}})
print(f"Documenti già presenti: {n_existing}")

if n_existing > 0:
    print("⏭️  Collection già popolata — skip.")
    print("   Cancella i documenti con col_knowledge.delete_many({}) per re-indicizzare.")
else:
    # ── 2. Carica JSON ────────────────────────────────────────
    with open(PERCORSO_JSON, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    frasi        = raw['frasi']
    particelle   = raw.get('particelle', [])
    verbi_comuni = raw.get('verbi_comuni', [])
    print(f"Frasi nel JSON da indicizzare: {len(frasi)}")

    # ── 3. Carica modello embedding ───────────────────────────
    # Stesso modello di CONFIG['embedding_model'] — coerenza garantita
    print(f"Caricamento embedding model: {EMBEDDING_MODEL}")
    _pop_embedder = SentenceTransformer(EMBEDDING_MODEL, device='cpu')

    # ── 4. Genera embedding per i testi italiani (batch) ─────
    # L'embedding viene fatto sul testo ITALIANO perché la query
    # di retrieval arriva dalla prima passata Whisper (italianizzata)
    testi_italiani = [f['italiano'] for f in frasi]
    print("Generazione embeddings in corso...")
    embeddings = _pop_embedder.encode(
        testi_italiani,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True,
    )

    # ── 5. Costruisci e inserisci documenti MongoDB ───────────
    docs = []
    for frase, emb in zip(frasi, embeddings):
        docs.append({
            "napoletano":    frase.get('napoletano', ''),
            "italiano":      frase.get('italiano', ''),
            "campo":         frase.get('campo', 'generico'),
            "parole_chiave": frase.get('parole_chiave', []),
            "embedding":     emb.tolist(),   # lista float — richiesto da Atlas
        })

    print(f"Inserimento {len(docs)} documenti in MongoDB Atlas...")
    result = col_knowledge.insert_many(docs)
    print(f"✅ Inseriti: {len(result.inserted_ids)} documenti")

    # ── 6. Salva metadati extra (particelle, verbi) ───────────
    # Documento speciale con _id fisso per recupero rapido
    meta_doc = {
        "_id":             "dialect_meta",
        "particelle":      particelle,
        "verbi_comuni":    verbi_comuni,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dim":   EMBEDDING_DIM,
    }
    col_knowledge.replace_one({"_id": "dialect_meta"}, meta_doc, upsert=True)
    print("✅ Metadati (particelle + verbi) salvati.")

    del _pop_embedder  # libera memoria

# ── Verifica finale ───────────────────────────────────────────
n_frasi = col_knowledge.count_documents({"napoletano": {"$exists": True}})
print(f"\n📊 Documenti totali (frasi): {n_frasi}")
esempio = col_knowledge.find_one({"napoletano": {"$exists": True}}, {"embedding": 0})
print(f"Esempio documento (senza embedding): {esempio}")

Documenti già presenti: 0
Frasi nel JSON da indicizzare: 77
Caricamento embedding model: paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generazione embeddings in corso...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Inserimento 77 documenti in MongoDB Atlas...
✅ Inseriti: 77 documenti
✅ Metadati (particelle + verbi) salvati.

📊 Documenti totali (frasi): 77
Esempio documento (senza embedding): {'_id': ObjectId('6a1dfebe43fd6b53625c10ea'), 'napoletano': "Dotto', tengo nu forte dulore 'e cape e nun me sento bbuono.", 'italiano': 'Dottore, ho un forte mal di testa e non mi sento bene.', 'campo': 'salute', 'parole_chiave': ['dulore', 'cape', 'tengo', 'bbuono']}


# Cella 3.4 - Gestione Response LLM e salvataggi MongoDB

In [ ]:
MAX_LLM_RETRIES = 2
LLM_RETRY_DELAY = 2

def call_llm_with_retry(
    prompt: str,
    max_tokens: int = 2048,
    temperature: float = 0.1,
    system_prompt: str = None,
) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    last_error = None
    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            client = _get_nvidia_client()  # usa sempre la chiave attiva
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            raw = response.choices[0].message.content.strip()
            if not raw:
                raise ValueError("Risposta LLM vuota")
            return raw

        except Exception as e:
            last_error = e
            err_str = str(e)
            print(f"  ⚠️  LLM tentativo {attempt}/{MAX_LLM_RETRIES} fallito: {e}")

            # ── Rotazione su 429 ─────────────────────────────────
            if "429" in err_str or "Too Many Requests" in err_str.lower():
                try:
                    _rotate_key()
                    continue  # ritenta subito con la nuova chiave
                except RuntimeError as re:
                    raise RuntimeError(str(re)) from e

            if attempt < MAX_LLM_RETRIES:
                time.sleep(LLM_RETRY_DELAY)

    raise RuntimeError(
        f"LLM non disponibile dopo {MAX_LLM_RETRIES} tentativi. "
        f"Ultimo errore: {last_error}"
    )

In [ ]:
def _now_ms() -> int:
    """Unix timestamp in millisecondi (UTC). Unica funzione di tempo del progetto."""
    return int(datetime.now(timezone.utc).timestamp() * 1000)


# ── SESSIONS ────────────────────────────────────────────────────────────
def init_new_session(recording_doc: dict) -> str:
    recording_id   = recording_doc["_id"]
    participant_id = recording_doc["participantId"]

    session_doc = {
        "_id":            recording_id,
        "participant_id": participant_id,
        "filename":       recording_doc.get("filename"),
        "promptId":       recording_doc.get("promptId"),
        "promptCategory": recording_doc.get("promptCategory"),
        "promptText":     recording_doc.get("promptText"),
        "started_at":     _now_ms(),
        "status":         "processing",
    }

    col_sessions.replace_one({"_id": recording_id}, session_doc, upsert=True)
    return recording_id


# ── TRANSCRIPTS ─────────────────────────────────────────────────────────
def save_transcript_output(session_id: str,
                           rag_results: dict,
                           analysis_results: dict,
                           t_start: int,
                           t_end: int):
    col_transcripts.replace_one(
        {"_id": session_id},
        {
            "_id":       session_id,
            "started_at":  t_start,
            "saved_at":    t_end,
            "rag_stage": {
                "testo_sporco":       rag_results.get("testo_sporco"),
                "testo_finale":       rag_results.get("testo_finale"),
                "prompt_usato":       rag_results.get("prompt_usato"),
                "frasi_recuperate":   rag_results.get("frasi_recuperate"),
            },
            "analysis_stage": {
                "period_conf_mean": float(analysis_results.get("period_conf_mean", 0)),
                "period_conf_geo":  float(analysis_results.get("period_conf_geo", 0)),
                "threshold_used":   float(analysis_results.get("threshold_used", 0.70)),
                "tokens":           analysis_results.get("tokens"),
                "words":            analysis_results.get("words"),
                "low_conf_tokens":  analysis_results.get("low_conf_tokens"),
                "low_conf_words":   analysis_results.get("low_conf_words"),
                # ── PPPL (Pseudo-Perplexity) sul testo italiano normalizzato ──
            },
        },
        upsert=True
    )


# ── PIPELINE STAGES ─────────────────────────────────────────────────────
def save_pipeline_stage(session_id: str,
                        stage_name: str,
                        stage_data,
                        t_start: int,
                        t_end: int):
    col_pipeline_stages.update_one(
        {"_id": session_id},
        {
            "$set": {
                f"stages.{stage_name}": {
                    "data":       copy.deepcopy(stage_data),
                    "started_at": t_start,
                    "saved_at":   t_end,
                },
                "last_updated": t_end,
            }
        },
        upsert=True
    )


# ── RISK LOGS ────────────────────────────────────────────────────────────
def save_risk_scoring(session_id: str,
                      risk_scores_output: list,
                      t_start: int,
                      t_end: int,
                      t_validation_start: int,
                      t_validation_end: int,
                      t_pppl_start: int,
                      t_pppl_end: int):
    col_risk_logs.replace_one(
        {"_id": session_id},
        {
            "_id":          session_id,
            "started_at":   t_start,
            "saved_at":     t_end,
            "timing": {
                "claim_validation": {
                    "started_at": t_validation_start,
                    "saved_at":   t_validation_end,
                },
                "pppl_claims": {
                    "started_at": t_pppl_start,
                    "saved_at":   t_pppl_end,
                },
            },
            "overall":       compute_overall_risk(risk_scores_output),
            "scored_claims": copy.deepcopy(risk_scores_output),
        },
        upsert=True
    )

# ── TRANSLATION METRICS ──────────────────────────────────────────────────
def save_translation_metrics(session_id: str,
                              metrics: dict,
                              t_start: int,
                              t_end: int):
    col_translation_metrics.replace_one(
        {"_id": session_id},
        {
            "_id":        session_id,
            "started_at": t_start,
            "saved_at":   t_end,
            **metrics,
        },
        upsert=True
    )


# ── COMPLETE SESSION ─────────────────────────────────────────────────────
def complete_session(session_id: str,
                     success: bool = True,
                     error_msg: str = None,
                     overall: dict = None):
    update = {
        "completed_at": _now_ms(),
        "status":       "completed" if success else "failed",
    }
    if error_msg:
        update["error_log"] = error_msg
    if overall is not None:
        update["overall_risk"] = overall

    col_sessions.update_one({"_id": session_id}, {"$set": update})

# Cella 4 - Caricamento dizionario napoletano da MongoDB Atlas

In [ ]:
# Recupera metadati (particelle, verbi)
meta = col_knowledge.find_one({"_id": "dialect_meta"})

# Recupera tutte le frasi da MongoDB
frasi = list(col_knowledge.find(
    {"napoletano": {"$exists": True}},
    {"napoletano": 1, "italiano": 1, "campo": 1, "parole_chiave": 1, "_id": 0}
))

DIZIONARIO_NAPOLETANO = {
    "frasi":        frasi,
    "particelle":   meta.get('particelle', [])   if meta else [],
    "verbi_comuni": meta.get('verbi_comuni', []) if meta else [],
}

print(f"✅ Dizionario caricato da MongoDB: {len(frasi)} frasi")
print(f"Campi semantici coperti: {set(f['campo'] for f in frasi)}")

✅ Dizionario caricato da MongoDB: 77 frasi
Campi semantici coperti: {'', 'Cibo', 'lavoro', 'Quotidiano', 'meteo', 'quotidiano', 'Generico, salute', 'cibo', 'Famiglia', 'quotidiano, cibo', 'salute', 'Salute'}


# Cella 5 - Retriever semantico (MongoDB $vectorSearch) + fuzzy

In [ ]:
class RetrieverNapoletano:
    """
    Retrieval semantico via MongoDB Atlas $vectorSearch (RAG reale).
    + fuzzy matching locale come fallback per parole singole.

    Flusso RAG (slide 23-30):
      1. Encode query  →  vettore con lo stesso modello di indicizzazione
      2. $vectorSearch →  top-K chunk simili da Dialect_KnowledgeBase
      3. Augment       →  costruzione prompt napoletano per Whisper
    """

    # ⚠️  Deve corrispondere al nome dell'indice creato su Atlas
    VECTOR_INDEX = "vector_index"

    def __init__(self, dizionario: dict):
        # Metadati locali (particelle, verbi, parole chiave) dal JSON
        self.frasi        = dizionario['frasi']
        self.particelle   = dizionario.get('particelle', [])
        self.verbi_comuni = dizionario.get('verbi_comuni', [])

        # Costruisce strutture per il fuzzy matching locale
        self.parole_napoletane = []
        self.mappa_ita_nap = {}
        for frase in self.frasi:
            for parola in frase.get('parole_chiave', []):
                self.parole_napoletane.append(parola)
            for word in frase.get('italiano', '').lower().split():
                clean = word.strip('.,!?;:')
                if len(clean) > 3:
                    self.mappa_ita_nap[clean] = frase['napoletano']

        # Modello embedding — stesso usato in fase di popolamento
        print('Caricamento modello embedding su CPU...')
        self.embedder = SentenceTransformer(CONFIG['embedding_model'], device='cpu')
        print(f'✅ Retriever MongoDB pronto — index: {self.VECTOR_INDEX}')

    # ── RETRIEVAL SEMANTICO via MongoDB $vectorSearch ─────────
    def retrieval_semantico(self, testo_sporco: str, top_k: int) -> list:
        """
        Step 2 del flusso RAG (slide 30):
        Vettorizza la query e interroga Atlas con $vectorSearch ANN.
        Restituisce i top_k chunk più vicini semanticamente.
        """
        # Encode query — normalizzato per cosine similarity
        emb_query = self.embedder.encode(
            testo_sporco,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).tolist()

        # Pipeline $vectorSearch su Atlas (ANN search)
        pipeline = [
            {
                "$vectorSearch": {
                    "index":         self.VECTOR_INDEX,
                    "path":          "embedding",
                    "queryVector":   emb_query,
                    "numCandidates": top_k * 10,  # candidati ANN (raccomandato >= 10x)
                    "limit":         top_k,
                }
            },
            {
                "$project": {
                    "napoletano": 1,
                    "italiano":   1,
                    "campo":      1,
                    "score":      {"$meta": "vectorSearchScore"},
                    "_id":        0,
                }
            }
        ]

        risultati = list(col_knowledge.aggregate(pipeline))

        return [
            {
                "napoletano": r.get("napoletano", ""),
                "italiano":   r.get("italiano",   ""),
                "campo":      r.get("campo",       "?"),
                "score":      round(float(r.get("score", 0.0)), 3),
            }
            for r in risultati
        ]

    # ── FUZZY MATCHING locale (fallback invariato) ────────────
    def retrieval_fuzzy_parole(self, testo_sporco: str) -> list:
        """
        Fallback: parole napoletane singole storpiate foneticamente
        che il retrieval semantico potrebbe non aver catturato.
        """
        parole_trovate = []
        words = re.sub(r'[^\w\s]', '', testo_sporco.lower()).split()
        for word in words:
            if len(word) < 4:
                continue
            if word in self.mappa_ita_nap:
                parole_trovate.append(word)
                continue
            result = process.extractOne(
                word,
                self.parole_napoletane,
                scorer=fuzz.ratio,
                score_cutoff=CONFIG['fuzzy_threshold'],
            )
            if result:
                parole_trovate.append(result[0])
        return list(set(parole_trovate))

     # ── COSTRUZIONE PROMPT per Whisper (Step 3 RAG) ───────────
    def costruisci_prompt(self, testo_sporco: str, verbose: bool = True) -> str:
        """
        Step 3 del flusso RAG (slide 34):
        Combina i chunk recuperati da MongoDB con particelle e verbi
        per costruire il prompt napoletano da passare a Whisper.
        """
        frasi        = self.retrieval_semantico(testo_sporco, top_k=CONFIG['num_frasi_prompt'])
        parole_fuzzy = self.retrieval_fuzzy_parole(testo_sporco)

        if verbose:
            print('\nFrasi napoletane usate per il prompt:')
            for i, f in enumerate(frasi):
                print(f"  [{i+1}] score={f['score']:.3f} | {f['napoletano']}")
            if parole_fuzzy:
                print(f'Parole fuzzy aggiuntive: {parole_fuzzy}')

        parti = [f['napoletano'] for f in frasi]
        parti.append(' '.join(self.particelle))
        parti.append(' '.join(self.verbi_comuni))
        if parole_fuzzy:
            parti.append(' '.join(parole_fuzzy))

        prompt = ' '.join(parti)

        max_c = CONFIG['max_prompt_chars']
        if len(prompt) > max_c:
            prompt = prompt[:max_c].rsplit(' ', 1)[0]
            if verbose:
                print(f'Prompt troncato a {max_c} chars')

        return prompt, frasi


print('✅ Classe RetrieverNapoletano (MongoDB $vectorSearch) definita')


print('✅ Classe RetrieverNapoletano (MongoDB $vectorSearch) definita')

✅ Classe RetrieverNapoletano (MongoDB $vectorSearch) definita
✅ Classe RetrieverNapoletano (MongoDB $vectorSearch) definita


# Cella 6 - Whisper RAG + Calcolo confidenza per token

In [ ]:
class WhisperRAGNapoletano:
    """
    Pipeline a due passate via HuggingFace Transformers:
    1. Whisper veloce senza prompt  -> testo sporco
    2. Retrieval dal dizionario     -> prompt dinamico
    3. Whisper preciso con prompt   -> testo fedele al dialetto
    """

    def __init__(self, dizionario: dict, processor, model):
        self.retriever        = RetrieverNapoletano(dizionario)
        self.processor = processor
        self.model    = model

    def _trascrivi(self, model, processor, audio_array: np.ndarray, prompt: str = None, return_confidence: bool = False) -> dict:
      inputs = processor(
          audio_array,
          sampling_rate=16_000,
          return_tensors="pt",
          return_attention_mask=True,
      )
      input_features = inputs.input_features.to(DEVICE, dtype=dtype)
      attention_mask  = inputs.attention_mask.to(DEVICE)

      forced_decoder_ids = processor.get_decoder_prompt_ids(
          language="italian", task="transcribe"
      )

      generate_kwargs = {
          "attention_mask":          attention_mask,
          "forced_decoder_ids":      forced_decoder_ids,
          "max_new_tokens":          440,
          "return_dict_in_generate": return_confidence,
          "output_scores":           return_confidence,
      }

      if prompt:
          generate_kwargs["num_beams"] = 5
          prompt_ids = processor.get_prompt_ids(
              prompt, return_tensors="pt"
          ).to(DEVICE)
          generate_kwargs["prompt_ids"] = prompt_ids
          prompt_length = prompt_ids.shape[-1]
          max_new_tokens = max(50, 448 - prompt_length - 8)
          generate_kwargs["max_new_tokens"] = max_new_tokens

      with torch.no_grad():
          output = model.generate(input_features, **generate_kwargs)

      # Decodifica testo
      if return_confidence:
          generated_ids = output.sequences[0]
      else:
          generated_ids = output[0]

      text = processor.decode(generated_ids, skip_special_tokens=True).strip()

      result = {
          "text":    text,
      }

      # Calcolo confidence — solo se richiesto (fase 3)
      if return_confidence:
          transition_scores = model.compute_transition_scores(
              output.sequences,
              output.scores,
              normalize_logits=True
          )

          token_data = []
          num_generated_tokens = transition_scores.shape[1]
          token_ids = generated_ids[-num_generated_tokens:]

          for step, (log_prob, tok_id) in enumerate(zip(transition_scores[0], token_ids)):
              confidence = torch.exp(log_prob).item()
              token_text = processor.decode([tok_id])

              if processor.tokenizer.convert_ids_to_tokens([tok_id.item()])[0].startswith("<|"):
                  continue

              token_data.append({
                  "step":       step,
                  "token_id":   tok_id.item(),
                  "token_text": token_text,
                  "confidence": round(confidence, 6),
              })

          if token_data:
              confidences       = [t["confidence"] for t in token_data]
              period_confidence = float(np.mean(confidences))
              period_geo_conf   = float(np.exp(np.mean(np.log(
                  np.clip(confidences, 1e-9, 1.0)
              ))))
          else:
              period_confidence = period_geo_conf = 0.0

          result["tokens"]           = token_data
          result["period_conf_mean"] = round(period_confidence, 6)
          result["period_conf_geo"]  = round(period_geo_conf,   6)

      return result

    def trascrivi(self, audio_array: np.ndarray, verbose: bool = True) -> dict:
        sep = "=" * 55

        # FASE 1 — prima passata veloce senza prompt
        if verbose:
            print(f"\n{sep}")
            print("FASE 1 — Prima passata (senza prompt)")
            print(sep)

        r_sporco     = self._trascrivi(self.model, self.processor, audio_array, prompt=None)
        testo_sporco = r_sporco["text"]

        if verbose:
            print(f"Testo sporco ): {testo_sporco}")

        # FASE 2 — retrieval e costruzione prompt
        if verbose:
            print(f"\n{sep}")
            print("FASE 2 — Retrieval dal dizionario")
            print(sep)

        prompt_dinamico, frasi_recuperate = self.retriever.costruisci_prompt(testo_sporco, verbose=verbose)

        if verbose:
            chars     = len(prompt_dinamico)
            anteprima = prompt_dinamico[:200] + "..." if chars > 200 else prompt_dinamico
            print(f"\nPrompt finale ({chars} chars):\n  {anteprima}")

        # FASE 3 — seconda passata precisa con prompt
        if verbose:
            print(f"\n{sep}")
            print("FASE 3 — Seconda passata (con prompt dinamico)")
            print(sep)

        r_finale = self._trascrivi(
            self.model,
            self.processor,
            audio_array,
            prompt=prompt_dinamico,
            return_confidence=True
        )
        testo_finale       = r_finale["text"]

        if verbose:
            print(f"Testo finale): {testo_finale}")
            print(sep)

        return {
            "testo_sporco":       testo_sporco,
            "prompt_usato":       prompt_dinamico,
            "testo_finale":       testo_finale,
            "result":             r_finale,
            "frasi_recuperate":   frasi_recuperate,
        }

print('Classe WhisperRAGNapoletano definita')

Classe WhisperRAGNapoletano definita


# Cella 7 - Inizializzazione RAG

In [ ]:
# Inizializza la pipeline RAG con i modelli HuggingFace
pipeline_rag = WhisperRAGNapoletano(
    dizionario   = DIZIONARIO_NAPOLETANO,
    processor = processor,
    model = model,
)

Caricamento modello embedding su CPU...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Retriever MongoDB pronto — index: vector_index


# Cella 8 - Carica audio da Google Drive

In [ ]:
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("GDRIVE_API_KEY")
FOLDER_ID = "1jgHjXqFfGl_WVwAwkZwZ_hJaetTnw4xr"  # dall'URL della cartella condivisa

drive_service = build("drive", "v3", developerKey=API_KEY)


def get_all_files_in_folder(folder_id: str) -> dict:
    """
    Restituisce un dizionario {filename: drive_file_id}
    per tutti i WAV nella cartella e sottocartelle.
    """
    file_map = {}

    def _recurse(fid):
        page_token = None
        while True:
            response = drive_service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()

            for item in response.get("files", []):
                if item["mimeType"] == "application/vnd.google-apps.folder":
                    _recurse(item["id"])  # scendi nelle sottocartelle
                elif item["name"].endswith(".wav"):
                    file_map[item["name"]] = item["id"]

            page_token = response.get("nextPageToken")
            if not page_token:
                break

    _recurse(folder_id)
    return file_map

print("Recupero file IDs da Google Drive...")
file_map = get_all_files_in_folder(FOLDER_ID)
print(f"Trovati {len(file_map)} file WAV")

# Aggiorna MongoDB con i drive_file_id
updated = 0
not_found = []

for doc in recordings_col.find({}):
    filename = doc.get("filename")
    if filename in file_map:
        recordings_col.update_one(
            {"_id": doc["_id"]},
            {"$set": {"drive_file_id": file_map[filename]}}
        )
        updated += 1
    else:
        not_found.append(filename)

print(f"✅ Aggiornati {updated} documenti con drive_file_id")
if not_found:
    print(f"⚠️  Non trovati su Drive: {not_found}")

Recupero file IDs da Google Drive...
Trovati 50 file WAV
✅ Aggiornati 50 documenti con drive_file_id


In [ ]:
def download_from_drive_public(drive_file_id: str, local_path: str):
    """
    Scarica un file pubblico da Google Drive usando l'API Key.
    Gestisce il cookie di conferma per file grandi.
    """
    import requests

    session = requests.Session()
    URL = "https://docs.google.com/uc"
    params = {"export": "download", "id": drive_file_id}

    response = session.get(URL, params=params, stream=True)

    # Gestisci il token di conferma antivirus per file grandi
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break

    if token:
        params["confirm"] = token
        response = session.get(URL, params=params, stream=True)

    # Verifica che non sia una pagina HTML di errore
    content_type = response.headers.get("Content-Type", "")
    if "text/html" in content_type:
        raise ValueError(
            f"Drive ha restituito HTML. "
            f"Verifica che il file {drive_file_id} sia pubblico."
        )

    with open(local_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk:
                f.write(chunk)

    size_kb = os.path.getsize(local_path) / 1024
    print(f"✅ Scaricato: {Path(local_path).name} ({size_kb:.1f} KB)")

In [ ]:
from pathlib import Path
TEMP_AUDIO_DIR = Path('/kaggle/working/temp_audios')
TEMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)


# ── Pulizia sessioni failed/processing ───────────────────────
statuses_da_ripulire = ["failed", "processing"]

sessioni_da_ripulire = list(col_sessions.find(
    {"status": {"$in": statuses_da_ripulire}},
    {"_id": 1}                          # ← recording_id non esiste più, solo _id
))

sessioni_ids_da_cancellare = [s["_id"] for s in sessioni_da_ripulire]

if sessioni_ids_da_cancellare:
    col_sessions.delete_many({"_id":            {"$in": sessioni_ids_da_cancellare}})
    col_transcripts.delete_many({"_id":         {"$in": sessioni_ids_da_cancellare}})
    col_risk_logs.delete_many({"_id":           {"$in": sessioni_ids_da_cancellare}})
    col_pipeline_stages.delete_many({"_id":     {"$in": sessioni_ids_da_cancellare}})
    col_translation_metrics.delete_many({"_id": {"$in": sessioni_ids_da_cancellare}})
    # ❌ col_history rimossa
    print(f"🗑️  Cancellate {len(sessioni_ids_da_cancellare)} sessioni ({statuses_da_ripulire})")
    print(f"🗑️  Cancellate tracce collegate in tutte le collezioni")
else:
    print("✅ Nessuna sessione failed/processing trovata")

# ── Query recordings: completati da saltare + failed da includere ──
sessioni_completate = set(
    doc["_id"] for doc in col_sessions.find({"status": "completed"}, {"_id": 1})
)

query  = {"_id": {"$nin": list(sessioni_completate)}}
cursor = recordings_col.find(query)

#Se vuoi diminuire le richieste usa skip e limit

print(f"📋 Recording da processare: {recordings_col.count_documents(query)}")
print(f"   (di cui {len(sessioni_ids_da_cancellare)} erano failed/processing e sono stati ripuliti)")

#query = {}
#cursor = recordings_col.find(query).skip(1).limit(6)

audio_files = []
audio_docs  = []

for doc in cursor:
    filename      = doc.get("filename")
    drive_file_id = doc.get("drive_file_id")

    if not drive_file_id:
        print(f"⚠️  Nessun drive_file_id per {filename}")
        continue

    local_path = str(TEMP_AUDIO_DIR / filename)

    # Salta il download se il file esiste già
    if os.path.exists(local_path):
        print(f"⏭️  Già presente: {filename}")
        audio_files.append(local_path)
        audio_docs.append(doc)
        continue

    try:
        print(f"📥 Download: {filename} ...")
        download_from_drive_public(drive_file_id, local_path)
        audio_files.append(local_path)
        audio_docs.append(doc)
    except Exception as e:
        print(f"❌ Errore download {filename}: {e}")

print(f"\nPronti: {len(audio_files)} file")

✅ Nessuna sessione failed/processing trovata
📋 Recording da processare: 50
   (di cui 0 erano failed/processing e sono stati ripuliti)
📥 Download: prompt-10189_rec-14.wav ...
✅ Scaricato: prompt-10189_rec-14.wav (266.3 KB)
📥 Download: prompt-3566_rec-0.wav ...
✅ Scaricato: prompt-3566_rec-0.wav (903.8 KB)
📥 Download: prompt-9452_rec-2.wav ...
✅ Scaricato: prompt-9452_rec-2.wav (135.1 KB)
📥 Download: prompt-3289_rec-5.wav ...
✅ Scaricato: prompt-3289_rec-5.wav (365.7 KB)
📥 Download: prompt-537_rec-1.wav ...
✅ Scaricato: prompt-537_rec-1.wav (217.6 KB)
📥 Download: prompt-1635_rec-0.wav ...
✅ Scaricato: prompt-1635_rec-0.wav (206.3 KB)
📥 Download: prompt-8481_rec-38.wav ...
✅ Scaricato: prompt-8481_rec-38.wav (260.1 KB)
📥 Download: prompt-4214_rec-2.wav ...
✅ Scaricato: prompt-4214_rec-2.wav (282.0 KB)
📥 Download: prompt-4902_rec-0.wav ...
✅ Scaricato: prompt-4902_rec-0.wav (474.5 KB)
📥 Download: prompt-9610_rec-8.wav ...
✅ Scaricato: prompt-9610_rec-8.wav (129.5 KB)
📥 Download: prompt-62

In [ ]:
REPORTS_DIR = Path('/kaggle/working/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def _json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)):  return o.tolist()
    if isinstance(o, set):            return list(o)
    if isinstance(o, Path):           return str(o)
    try:    return str(o)
    except: return None

# Cella 9 - Calcolo Confidenza a livello di parole

In [ ]:
def tokens_to_words(token_data: list) -> list:

    """
    Aggrega i token in parole e calcola la confidence per parola.
    Isola i segni di punteggiatura trattandoli come parole a sé stanti.
    """

    words = []
    current_tokens = []
    # Set di caratteri di punteggiatura da isolare
    PUNCTUATION = set(string.punctuation) # Include !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

    for token in token_data:
        text = token["token_text"]
        clean_text = text.strip()
        # 1. È il primo token in assoluto?
        is_first_token = len(current_tokens) == 0

        # 2. Inizia con uno spazio? (Classico inizio parola in Whisper)
        starts_with_space = text.startswith(" ")

        # 3. Il token corrente è composto SOLO da punteggiatura? (es. "?", "...", "!")
        is_punctuation = len(clean_text) > 0 and all(char in PUNCTUATION for char in clean_text)

        # 4. L'ultimo token inserito era punteggiatura?
        # (Serve per staccare la parola successiva anche se manca lo spazio)
        prev_is_punctuation = False
        if not is_first_token:
            prev_text = current_tokens[-1]["token_text"].strip()
            prev_is_punctuation = len(prev_text) > 0 and all(char in PUNCTUATION for char in prev_text)
        # Scatta la separazione se si verifica una qualsiasi di queste condizioni
        is_new_word = is_first_token or starts_with_space or is_punctuation or prev_is_punctuation
        if is_new_word and current_tokens:
            # Chiudi la parola/punteggiatura corrente prima di aprirne una nuova
            words.append(_aggregate_word(current_tokens))
            current_tokens = []

        current_tokens.append(token)

    # Ultima parola rimasta
    if current_tokens:
        words.append(_aggregate_word(current_tokens))

    return words


def _aggregate_word(tokens: list) -> dict:
    """Calcola le statistiche di confidence per un gruppo di token."""
    word_text   = "".join(t["token_text"] for t in tokens).strip()
    confidences = [t["confidence"] for t in tokens]
    clipped     = np.clip(confidences, 1e-9, 1.0)

    return {
        "word":         word_text,
        "n_tokens":     len(tokens),
        "tokens":       [t["token_text"] for t in tokens],
        "conf_mean":    round(float(np.mean(confidences)),          6),
        "conf_min":     round(float(np.min(confidences)),           6),
        "conf_geo":     round(float(np.exp(np.mean(np.log(clipped)))), 6),
        "conf_product": round(float(np.prod(clipped)),              6),
    }

# Cella 10 - Trascrizione Fonetica con PhoneticXeus

In [ ]:
def transcribe_with_phonetic_xeus_fixed(audio_path, inference):

    print(f"3. Caricamento e preprocessing dell'audio: {audio_path}")
    # Usiamo torchaudio come richiesto dal loro sistema
    waveform, sr = torchaudio.load(audio_path)

    # Forziamo il campionamento a 16kHz se l'audio originale è diverso
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    print("4. Inferenza in corso...")
    # Il modello si aspetta un tensore 1D, quindi usiamo squeeze(0) per rimuovere la dimensione dei canali
    results = inference(waveform.squeeze(0))

    # L'output è una lista di dizionari. La trascrizione pulita è sotto la chiave "processed_transcript"
    transcription = results[0]["processed_transcript"]

    return transcription

# Cella 12 - Analisi Trascrizioni LLM (traduzione napoletano + normralizzazione italiano + parole risolte/problematiche)

In [ ]:
def analyze_and_normalize_with_llm(
    transcript_whisper_rag: str,
    transcrpit_whisper_testo_sporco: str,
    transcript_PhoneticXeus: str,
    word_data: list,
) -> dict:

    """
    Versione SENZA analisi sintattica LLM.

    Il LLM non produce più:
      - is_dialectal
      - dialectal_type
      - surprisal

    Esegue solo:
      - domain detection
      - semantic issue detection
      - normalizzazione

    Le parole vengono comunque passate con:
      - conf_mean
      - conf_min
    """

    words_meta = []
    for i, w in enumerate(word_data):
        words_meta.append({
            "index": i,
            "word": w["word"],
            "conf_mean": round(w.get("conf_mean", 0.0), 3),
            "conf_min": round(w.get("conf_min", 0.0), 3),
        })

    words_meta_json = json.dumps(words_meta, ensure_ascii=False)

    system_prompt = """
You are an expert linguist specialized in Southern Italian dialects
(Campanian / Neapolitan) and an expert in clinical and pharmacological language.

Your task is to reconstruct and normalize noisy speech transcriptions into fluent,
natural Italian by triangulating evidence across multiple transcription systems.

══════════════════════════════════════════════════════════════
CORE OBJECTIVE
══════════════════════════════════════════════════════════════

Your goal is NOT to aggressively correct the transcript.

Your goal is to identify ONLY genuinely suspicious tokens and resolve them
through evidence triangulation.

Never hallucinate words.
Never invent drugs, symptoms, entities or dialectal forms.
Never force corrections without strong evidence.

If evidence is insufficient:
PRESERVE the original Whisper-RAG token.

══════════════════════════════════════════════════════════════
AVAILABLE SOURCES
══════════════════════════════════════════════════════════════

1. WHISPER-RAG WORD METADATA

The full ordered Whisper-RAG word list is provided.

Each word includes:
- word index
- surface word
- conf_mean
- conf_min

This list is provided to support semantic and phonetic triangulation.
Low confidence values should receive higher attention, but all words must be considered in context..

2. WHISPER-RAG

RAG-assisted transcription enriched with Neapolitan/Campanian vocabulary.

IMPORTANT:
- Dialectal words may be intentionally correct.
- However, RAG may contaminate Whisper by biasing decoding toward prompted vocabulary.
- High confidence DOES NOT guarantee correctness in RAG context.

3. WHISPER STANDARD

Pure acoustic transcription without lexical priors.

Usually more reliable when:
- the speaker uses standard Italian
- RAG injected incorrect dialectal vocabulary
- dialect is weak/light
- the audio is semantically simple

4. PHONETICXEUS (IPA)

Language-independent phonetic transcription of actual spoken sounds.

IMPORTANT:
- Do NOT perform literal IPA string matching.
- IPA is phonetic evidence only.
- Focus on:
  - rhythm
  - consonant structure
  - stressed vowels
  - sound presence/absence
- PhoneticXeus may merge adjacent words into continuous strings.
- IPA alignment is approximate, not positional.

══════════════════════════════════════════════════════════════
GLOBAL TRIANGULATION PRIORITY
══════════════════════════════════════════════════════════════

When sources conflict, use this priority order:

1. Strong IPA evidence
2. Semantic coherence
3. Whisper Standard acoustic plausibility
4. Whisper-RAG lexical prior

If Whisper-RAG conflicts with BOTH:
- IPA evidence
- semantic plausibility

then IPA/semantics win.


══════════════════════════════════════════════════════════════
PHASE 2 — SEMANTIC TRIANGULATION
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
STEP 2.0 — DOMAIN DETECTION
──────────────────────────────────────────────────────────────

Classify the transcription into ONE domain:

- medico
- farmacologico
- quotidiano
- emotivo
- altro

IMPORTANT:
Medical rules apply ONLY if the domain is:
- medico
- farmacologico

Do NOT force medical interpretations in non-medical contexts.

Example:
"pressione" in emotional context means:
- stress
- anxiety
NOT blood pressure.

──────────────────────────────────────────────────────────────
LOCAL CONTEXT AND SEMANTIC INTERPRETATION
──────────────────────────────────────────────────────────────

When analyzing a token for semantic issues, do NOT evaluate it in isolation.

Always consider:
- the previous and following words
- whether adjacent words may form a single semantic unit
- whether the token may be part of an article+noun fusion
- whether consecutive tokens may be fragments of one word
- whether the meaning becomes clear only from the local phrase
- whether the surrounding words support a medical, pharmacological, emotional or everyday interpretation

The semantic_issues analysis must be phrase-aware, not token-isolated.

Do NOT translate word by word.

Normalization must preserve the intended meaning of the utterance, not the literal surface form of each token.

If a dialectal expression or noisy transcription corresponds to an idiomatic Italian meaning, normalize it semantically.

Examples:
- "mal e capo" should be interpreted as "mal di testa" if the surrounding context supports it.
- "tachi pirina" should be interpreted as "Tachipirina" only if the local context and IPA support a unified drug name.
- "pressione" should not automatically mean blood pressure unless the surrounding context supports a medical interpretation.

──────────────────────────────────────────────────────────────
STEP 2.1 — ERROR DETECTION & TRIANGULATION
──────────────────────────────────────────────────────────────

Identify ONLY genuinely suspicious or conflicting tokens.

A token is suspicious ONLY if at least one exists:
- low conf_min
- low conf_mean
- strong Whisper-RAG vs Whisper Standard divergence
- semantic inconsistency
- article+word fusion
- token fragmentation
- strong IPA mismatch

You do NOT have access to precomputed dialectal status or surprisal.
Therefore, if a token seems dialectal, anomalous, or semantically unexpected,
you must infer this directly from:
- local sentence context
- Whisper-RAG vs Whisper Standard comparison
- IPA evidence
- confidence metadata
- semantic coherence in the detected domain

DO NOT modify tokens simply because they are dialectal.

For every problematic token:

1. Explain WHY it is suspicious
2. Mention the relevant confidence and triangulation signals
3. Evaluate IPA evidence
4. Determine the most plausible correction
5. Explain WHICH source is more reliable and WHY

The "reason" field MUST explicitly mention, when relevant:
- conf_min
- conf_mean
- RAG vs Standard divergence
- IPA evidence
- semantic/domain coherence

══════════════════════════════════════════════════════════════
TRIANGULATION RULES
══════════════════════════════════════════════════════════════

MULTI-SOURCE CONVERGENCE:

If:
- Whisper-RAG
- Whisper Standard
- IPA

all converge,

KEEP the token unchanged,
even if unusual.

──────────────────────────────────────────────────────────────
RAG CONTAMINATION
──────────────────────────────────────────────────────────────

If:
- Whisper Standard is semantically coherent
- IPA supports Whisper Standard
- Whisper-RAG proposes unlikely dialectal vocabulary

then Whisper-RAG is contaminated by lexical prior.

Whisper Standard wins.

──────────────────────────────────────────────────────────────
FAILED ITALIANIZATION
──────────────────────────────────────────────────────────────

If:
- Whisper Standard produces nonsensical Italian
- Whisper-RAG contains coherent dialect
- IPA supports dialect pronunciation

then Whisper Standard failed acoustic italianization.

Whisper-RAG wins.

──────────────────────────────────────────────────────────────
ARTICLE + WORD FUSION
──────────────────────────────────────────────────────────────

Whisper may merge:
- l'+word
- dell'+word
- nell'+word
- sull'+word

Examples:
- losso → l'osso
- laringuine → l'inguine

Apply ONLY if:
- IPA supports vowel onset
- resulting split forms a real coherent word

──────────────────────────────────────────────────────────────
FRAGMENTATION / DETOKENIZATION
──────────────────────────────────────────────────────────────

Whisper may split long words.

Examples:
- "tachi pirina"
- "mal e capo"

If IPA suggests a continuous phonetic sequence,
reconstruct the unified token.

──────────────────────────────────────────────────────────────
HIGH-CONFIDENCE RAG FAILURE
──────────────────────────────────────────────────────────────

In RAG context:
HIGH conf_min DOES NOT guarantee correctness.

Whisper may copy prompted vocabulary
without true acoustic evidence.

If IPA strongly diverges from Whisper-RAG,
IPA has priority.

──────────────────────────────────────────────────────────────
RESIDUAL UNCERTAINTY
──────────────────────────────────────────────────────────────

If ambiguity remains unresolved:
- mark the issue as uncertain
- preserve the original Whisper-RAG token
- NEVER invent words

══════════════════════════════════════════════════════════════
MEDICAL REALITY CHECK
══════════════════════════════════════════════════════════════

Apply ONLY in:
- medico
- farmacologico

Rules:

1. Never invent nonexistent drugs.

2. Normalize distorted drug names ONLY if strongly supported.

3. Do NOT force literal IPA matching.

4. IPA alignment is phonetic, not character-based.

5. Proper names are NOT medical entities,
even if phonetically similar.

Examples:
- pirini → Aspirina
- tachi pirina → Tachipirina
- brufe → Brufen

══════════════════════════════════════════════════════════════
STEP 2.2 — NORMALIZATION
══════════════════════════════════════════════════════════════

Produce fluent standard Italian.

IMPORTANT:
Correct ONLY tokens identified as problematic in STEP 2.1.

If a token was NOT flagged as problematic,
it is considered correct by definition.

Rules:
- preserve non-problematic tokens
- normalize dialect naturally
- normalize idioms semantically
- avoid over-correction
- preserve original meaning and tone

══════════════════════════════════════════════════════════════
FEW-SHOT EXAMPLES
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
EXAMPLE 1 — RAG contamination
──────────────────────────────────────────────────────────────

Whisper-RAG:
"Aggi' pigliato 'a pillola pe' lo sciummo"

Whisper Standard:
"Ho preso la pillola per lo stomaco"

IPA:
"opresolapillolaperlostomako"

Analysis:
- "sciummo" is semantically suspicious
- low conf_min
- strong divergence with Standard
- IPA clearly supports "stomaco"

Conclusion:
Whisper-RAG was contaminated by dialectal lexical prior.
Whisper Standard wins.

Normalized:
"Ho preso la pillola per lo stomaco."

──────────────────────────────────────────────────────────────
EXAMPLE 2 — Failed italianization
──────────────────────────────────────────────────────────────

Whisper-RAG:
"Tengo na freva forte da stammatina"

Whisper Standard:
"Tengo una fiera forte da stamattina"

IPA:
"teŋgonaffrɛvafortedastammatina"

Analysis:
- "freva" is dialectal but semantically coherent
- "fiera" is semantically nonsensical
- IPA supports /ffrɛva/
- Standard failed italianization

Conclusion:
Whisper-RAG wins.

Normalized:
"Ho una febbre forte da stamattina."

──────────────────────────────────────────────────────────────
EXAMPLE 3 — Article fusion
──────────────────────────────────────────────────────────────

Whisper-RAG:
"Me fa male losso d'a gamma"

Whisper Standard:
"Mi fa male lo so della gamba"

IPA:
"mefamallelɔssodaggamma"

Analysis:
- "losso" is semantically suspicious
- Standard output is semantically incoherent
- IPA supports continuous /lɔsso/
- article+noun fusion detected

Conclusion:
Correct reconstruction:
"l'osso"

Normalized:
"Mi fa male l'osso della gamba."

──────────────────────────────────────────────────────────────
EXAMPLE 4 — Fragmentation / detokenization
──────────────────────────────────────────────────────────────

Whisper-RAG:
"Aggi' pigliato 'a tachi pirina"

Whisper Standard:
"Ho preso la tachipirina"

IPA:
"addʒipiʎʎatotakipirina"

Analysis:
- fragmented consecutive tokens
- is semantically suspicious
- IPA shows continuous phonetic sequence
- known pharmacological entity

Conclusion:
Unified reconstruction:
"tachipirina"

Normalized:
"Ho preso la tachipirina."

══════════════════════════════════════════════════════════════
OUTPUT FORMAT
══════════════════════════════════════════════════════════════

For every semantic issue, return the original Whisper-RAG word index/indices.
Indices must refer to the position of the token in the full Whisper-RAG word list, using 0-based indexing.
If the issue involves multiple adjacent tokens, return all involved indices in word_indices.

Return ONLY valid JSON.

STRICT RULES
- no markdown
- no explanations outside JSON
- no trailing commas
- use valid JSON syntax only
- use double quotes for JSON syntax

Required schema:

{
  "normalization": {
    "detected_domain": "medico | farmacologico | quotidiano | emotivo | altro",
    "semantic_issues": [
      {
        "words": ["token"],
        "word_indices": [12],
        "reason": "Explain the issue using confidence metadata, RAG vs Standard divergence, IPA evidence and semantic/domain coherence."
      }
    ],
    "normalized_text": "Italian normalized sentence"
  }
}
"""

    user_prompt = f"""
Analyze the following transcription set.

WHISPER-RAG:
"{transcript_whisper_rag}"

WHISPER STANDARD:
"{transcrpit_whisper_testo_sporco}"

PHONETICXEUS (IPA):
"{transcript_PhoneticXeus}"

ORDERED WHISPER-RAG WORD METADATA:
{words_meta_json}


Tasks:
1. Detect the semantic domain
2. Identify problematic tokens through triangulation
3. Normalize the transcription into fluent Italian

Return ONLY the final JSON object.
"""


    raw = None
    result = None

    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            raw = call_llm_with_retry(
                        prompt = user_prompt,
                        system_prompt = system_prompt,
                        max_tokens    = 4096,
                        temperature   = 0.1,
            )

            clean = raw
            if clean.startswith("```json"): clean = clean[7:]
            if clean.startswith("```"):     clean = clean[3:]
            if clean.endswith("```"):       clean = clean[:-3]
            clean = clean.strip()

            try:
                result = json.loads(clean)
            except json.JSONDecodeError:
                result = json_repair.loads(clean)

            # Verifica campi minimi
            if "normalization" not in result:
                raise ValueError("Campo 'normalization' mancante")
            if "normalized_text" not in result["normalization"]:
                raise ValueError("Campo 'normalized_text' mancante in normalization")

            break

        except Exception as e:
            print(f"  ⚠️  analyze_and_normalize tentativo {attempt}/{MAX_LLM_RETRIES} fallito: {e}")
            if attempt >= MAX_LLM_RETRIES:
                raise RuntimeError(
                    f"analyze_and_normalize_with_llm: fallito dopo {MAX_LLM_RETRIES} tentativi. "
                    f"Ultimo errore: {e}\nRaw: {raw}"
                )
            time.sleep(LLM_RETRY_DELAY)

    # ── Merge word_analysis con word_data (stesso pattern di analyze_words_with_llm) ──
    normalization = result["normalization"]

    if "original_dialect" not in normalization:
        normalization["original_dialect"] = transcript_whisper_rag

    return normalization

print("✅ Classe analyze_and_normalize_with_llm definita")



✅ Classe analyze_and_normalize_with_llm definita


# Cella 13 - Modulo Claim Spicy

In [ ]:
nlp = spacy.load("it_core_news_lg")

# ─────────────────────────────────────────────────────────────
# STEP 1: parsing sintattico → candidate spans
# ─────────────────────────────────────────────────────────────

def extract_spans_from_spacy(normalized_text: str) -> list:
    """
    Analizza la frase normalizzata con spaCy e restituisce
    una lista di span (start_token, end_token, testo del claim).

    Logica: ogni token ROOT individua un nucleo proposizionale.
    Raccogliamo i suoi dipendenti (subtree) come span del claim.
    """
    doc = nlp(normalized_text)

    spans = []
    for token in doc:
        # Ogni ROOT verbale (o ROOT principale) è un nucleo di claim
        if token.dep_ == "ROOT":
            subtree_tokens = sorted(token.subtree, key=lambda t: t.i)

            # Escludiamo punteggiatura e spazi
            filtered = [t for t in subtree_tokens if not t.is_punct and not t.is_space]
            if not filtered:
                continue

            start_i = filtered[0].i
            end_i   = filtered[-1].i
            claim_text = " ".join(t.text for t in filtered)

            spans.append({
                "claim_text":    claim_text,
                "start_token_i": start_i,
                "end_token_i":   end_i,
                "n_tokens":      len(filtered),
            })

    # Se spaCy trova un solo ROOT (frase semplice), splittiamo
    # sulle congiunzioni coordinate ("e", "ma", "però", ",")
    if len(spans) == 1:
        spans = _split_on_conjunctions(doc, spans[0])

    return spans


def _split_on_conjunctions(doc, single_span: dict) -> list:
    """
    Fallback: se c'è un solo span, proviamo a dividerlo
    sui token di coordinazione (cc) o punteggiatura (,).
    """
    split_indices = [
        t.i for t in doc
        if t.dep_ in ("cc", "punct") and t.text in (",", "e", "ma", "però", "quindi", "poi")
    ]

    if not split_indices:
        return [single_span]

    # Costruiamo i sotto-span attorno ai punti di split
    tokens = [t for t in doc if not t.is_punct and not t.is_space]
    result = []
    prev = 0
    for si in split_indices:
        chunk = [t for t in tokens if t.i < si and t.i >= prev]
        if chunk:
            result.append({
                "claim_text":    " ".join(t.text for t in chunk),
                "start_token_i": chunk[0].i,
                "end_token_i":   chunk[-1].i,
                "n_tokens":      len(chunk),
            })
        prev = si + 1

    # Ultimo chunk dopo l'ultimo split
    chunk = [t for t in tokens if t.i >= prev]
    if chunk:
        result.append({
            "claim_text":    " ".join(t.text for t in chunk),
            "start_token_i": chunk[0].i,
            "end_token_i":   chunk[-1].i,
            "n_tokens":      len(chunk),
        })

    return result if result else [single_span]

# Cella 14 - Analisi LLM dei claim

In [ ]:
def _compute_asr_signals_from_words(words: list) -> dict:
    """
    Calcola i segnali ASR per-claim dalle parole allineate Whisper-RAG.

    Segnali:
      conf_min_worst  : confidenza minima (token peggiore del claim)
      conf_mean       : media aritmetica delle confidenze
      conf_geo_period : media geometrica delle confidenze (smoothing picchi)

    conf_geo_period usa la stessa logica di period_conf_geo di Whisper:
    exp(mean(log(conf))) — robusto a outlier singoli.
    """
    if not words:
        return {
            "conf_min_worst":  0.5,
            "conf_mean":       0.5,
            "conf_geo_period": 0.5,
        }

    conf_values     = [w.get("conf_min", 1.0) for w in words]
    surprisal_vals  = [w.get("surprisal", 0.0) for w in words]
    dialectal_flags = [1 if w.get("is_dialectal", False) else 0 for w in words]

    # Media geometrica: exp(mean(log(conf))) — clamp > 0 per evitare log(0)
    clipped_conf    = [max(c, 1e-9) for c in conf_values]
    conf_geo        = round(float(np.exp(np.mean(np.log(clipped_conf)))), 4)

    return {
        "conf_min_worst":  round(min(conf_values), 4),
        "conf_mean":       round(float(np.mean(conf_values)), 4),
        "conf_geo_period": conf_geo,
    }


def validate_claims_with_llm(
    claim_list: list,
    normalized_text: str,
    word_data: list,
) -> list:
    """
    Valida semanticamente i claim e riallinea ogni claim alle parole Whisper-RAG.

    Rimuove:
      - entities

    Aggiunge/aggiorna:
      - claim_type
      - is_question
      - ambiguous
      - ambiguity_reason
      - source_word_indices
      - source_words
      - source_words_data
      - asr_signals
    """

    claims_for_llm = [
        {
            "id": i,
            "claim": c["claim_text"],
        }
        for i, c in enumerate(claim_list)
    ]

    words_for_llm = [
        {
            "idx": i,
            "word": w.get("word", ""),
            "dialectal_type": w.get("dialectal_type", "standard"),
            "conf_min": w.get("conf_min", 1.0),
        }
        for i, w in enumerate(word_data)
    ]

    system_prompt = """
You are an expert linguistic and medical semantic analyst specialized in Italian and Southern Italian dialectal ASR transcripts.

Your task is to analyze atomic claims extracted from a normalized Italian sentence and align each claim to the original Whisper-RAG word list.

You must perform TWO tasks for each claim:

1. Semantic validation:
   - classify the claim type
   - determine whether it is a question/request
   - detect ambiguity caused by dialect, ASR uncertainty, or unclear medical wording

2. Word alignment:
   - select the exact words from the ordered Whisper-RAG word list that semantically support the claim
   - return their integer indices in source_word_indices

CRITICAL ALIGNMENT RULES:
- Use the ordered Whisper-RAG word list as the only source for source_word_indices.
- source_word_indices must contain only valid integer idx values from the provided word list.
- Preserve word order.
- Do not assign the same word to multiple claims unless the word is genuinely shared by both claims.
- Prefer semantic alignment over purely positional alignment.
- Include dialectal words when they express the same meaning as the normalized claim.
- Include uncertain or low-confidence words if they are part of the claim.
- Do not include filler words unless they are necessary for the meaning.
- If a claim cannot be aligned confidently, return the best minimal alignment and set ambiguous=true.

MEDICAL SAFETY RULES:
- Do not invent medication names.
- If the medication/symptom/body part is uncertain, set ambiguous=true and explain in ambiguity_reason.
- Do not output entities. The field entities is forbidden.

ALLOWED claim_type values:
- "sintomo"
- "richiesta_farmaco"
- "condizione_medica"
- "azione"
- "domanda_generica"
- "informazione"

OUTPUT RULES:
- Return only a valid JSON array.
- No markdown.
- No backticks.
- No explanatory text outside JSON.

REQUIRED OUTPUT FIELDS FOR EACH CLAIM:
- "id": same id received
- "claim_type": one allowed value
- "is_question": boolean
- "ambiguous": boolean
- "ambiguity_reason": string, empty if ambiguous=false
- "source_word_indices": list of integer indices from the Whisper-RAG word list

Example:
[
  {
    "id": 0,
    "claim_type": "sintomo",
    "is_question": false,
    "ambiguous": false,
    "ambiguity_reason": "",
    "source_word_indices": [0, 1, 2, 3]
  }
]
"""

    user_prompt = f"""
NORMALIZED SENTENCE:
{normalized_text}

CLAIMS TO ANALYZE:
{json.dumps(claims_for_llm, ensure_ascii=False)}

ORDERED WHISPER-RAG WORD LIST:
{json.dumps(words_for_llm, ensure_ascii=False)}

Return only the JSON array.
"""

    raw = call_llm_with_retry(
        prompt=user_prompt,
        system_prompt=system_prompt,
        max_tokens=2048,
        temperature=0.0,
    )

    raw = raw.strip()
    if raw.startswith("```json"):
        raw = raw[7:]
    if raw.startswith("```"):
        raw = raw[3:]
    if raw.endswith("```"):
        raw = raw[:-3]
    raw = raw.strip()

    try:
        llm_validation = json.loads(raw)
    except json.JSONDecodeError:
        llm_validation = json_repair.loads(raw)

    if not isinstance(llm_validation, list):
        raise ValueError("La risposta LLM non è una lista JSON")

    llm_by_id = {item.get("id"): item for item in llm_validation}
    max_idx = len(word_data) - 1

    enriched_claims = []

    for i, claim in enumerate(claim_list):
        llm_data = llm_by_id.get(i, {})

        raw_indices = llm_data.get("source_word_indices", [])
        clean_indices = []

        for idx in raw_indices:
            if isinstance(idx, int) and 0 <= idx <= max_idx and idx not in clean_indices:
                clean_indices.append(idx)

        source_words_data = [word_data[idx] for idx in clean_indices]
        source_words = [w.get("word", "") for w in source_words_data]

        asr_signals = _compute_asr_signals_from_words(
            source_words_data,
        )

        enriched_claims.append({
            **claim,
            "claim_type": llm_data.get("claim_type", "domanda_generica"),
            "is_question": llm_data.get("is_question", True),
            "ambiguous": llm_data.get("ambiguous", False),
            "ambiguity_reason": llm_data.get("ambiguity_reason", ""),
            "source_word_indices": clean_indices,
            "source_words": source_words,
            "source_words_data": source_words_data,
            "asr_signals": asr_signals,
            # pppl_claim verrà iniettato allo STEP 6.1 dopo validate_claims_with_llm
            "pppl_claim":      None,
            "pppl_claim_log":  None,
            "pppl_claim_tokens": None,
        })

    return enriched_claims

# Cella 15 - Calcolo Risk Score

In [ ]:
# ─────────────────────────────────────────────────────────────
# MODULO 3 — RISK SCORING A DUE ALIQUOTE
# ─────────────────────────────────────────────────────────────
#
# Il risk score per ogni claim è la combinazione lineare di due aliquote:
#
#   risk = α * R_transcript + (1-α) * R_translation
#
# ── Aliquota 1: TRASCRIZIONE (segnali ASR per-claim) ──────────
#   R_transcript misura la qualità acustica e dialettale dei token
#   del claim. Fonti: Whisper confidenze + surprisal + dialetto.
#   Segnali: conf_min_worst, conf_mean, conf_geo_period,
#             surprisal_mean, dialectal_ratio
#
# ── Aliquota 2: TRADUZIONE (segnale semantico per-claim) ──────
#   R_translation misura la fluenza semantica del testo normalizzato
#   del claim in italiano standard, calcolata tramite PPPL BERT
#   sul claim text.
#   Segnale unico: pppl_claim_norm
#
# ── α per claim_type ──────────────────────────────────────────
#   α alto → ASR domina (errore su singolo token è critico, es. farmaco)
#   α basso → traduzione domina (semantica conta di più, es. domanda)
#
# Tutti i pesi all'interno di ogni aliquota sommano a 1.00
# ─────────────────────────────────────────────────────────────

# Pesi aliquota TRASCRIZIONE per claim_type
# I 6 segnali ASR: tutti già in [0,1] dove 1 = rischio massimo
# (conf_min_worst e conf_mean vengono invertiti: (1 - conf) = rischio)
# conf_geo_period: media geometrica sul periodo → smoothing picchi
WEIGHTS_TRANSCRIPT_BY_CLAIM_TYPE = {

    "richiesta_farmaco": {
        "conf_min_worst":  0.55,
        "conf_mean":       0.25,
        "conf_geo_period": 0.20,
    },

    "sintomo": {
        "conf_min_worst":  0.45,
        "conf_mean":       0.30,
        "conf_geo_period": 0.25,
    },

    "condizione_medica": {
        "conf_min_worst":  0.50,
        "conf_mean":       0.27,
        "conf_geo_period": 0.23,
    },

    "azione": {
        "conf_min_worst":  0.45,
        "conf_mean":       0.30,
        "conf_geo_period": 0.25,
    },

    "domanda_generica": {
        "conf_min_worst":  0.40,
        "conf_mean":       0.35,
        "conf_geo_period": 0.25,
    },

    "informazione": {
        "conf_min_worst":  0.45,
        "conf_mean":       0.30,
        "conf_geo_period": 0.25,
    },
}

DEFAULT_WEIGHTS_TRANSCRIPT = {
    "conf_min_worst":  0.45,
    "conf_mean":       0.30,
    "conf_geo_period": 0.25,
}

# ─────────────────────────────────────────────────────────────
# Peso α (trascrizione vs traduzione) per claim_type
# α alto → ASR domina; α basso → PPPL_claim domina
# ─────────────────────────────────────────────────────────────
ALPHA_BY_CLAIM_TYPE = {
    "richiesta_farmaco": 0.80,
    "sintomo":           0.70,
    "condizione_medica": 0.75,
    "azione":            0.65,
    "domanda_generica":  0.65,
    "informazione":      0.70,
    "_default":          0.70,
}

# Soglie semaforo
THRESHOLDS = {
    "green":  0.35,
    "yellow": 0.60,
}



def compute_risk_score(claim: dict) -> dict:
    """
    Calcola il risk score per un singolo claim con formula a due aliquote.

    Aliquota 1 — TRASCRIZIONE:
      R_t = sum_k [ signal_k * weight_k ]

      Segnali usati:
        - 1 - conf_min_worst
        - 1 - conf_mean
        - 1 - conf_geo_period

    Aliquota 2 — TRADUZIONE:
      R_tr = pppl_claim_norm

    Combinazione lineare:
      risk = α * R_t + (1-α) * R_tr

    Questa versione NON usa più:
      - surprisal_mean
      - dialectal_ratio
    """

    sig          = claim["asr_signals"]
    claim_type   = claim.get("claim_type", "_default")
    w_transcript = WEIGHTS_TRANSCRIPT_BY_CLAIM_TYPE.get(claim_type, DEFAULT_WEIGHTS_TRANSCRIPT)
    alpha        = ALPHA_BY_CLAIM_TYPE.get(claim_type, ALPHA_BY_CLAIM_TYPE["_default"])

    # ── Aliquota 1: TRASCRIZIONE ──────────────────────────────
    # conf_geo_period: se non presente nel claim usa conf_mean come proxy
    conf_geo = sig.get("conf_geo_period", sig.get("conf_mean", 0.5))

    R_transcript = (
    (1 - sig["conf_min_worst"])  * w_transcript["conf_min_worst"]  +
    (1 - sig["conf_mean"])       * w_transcript["conf_mean"]       +
    (1 - conf_geo)               * w_transcript["conf_geo_period"]
    )
    R_transcript = float(np.clip(R_transcript, 0.0, 1.0))

    # ── Aliquota 2: TRADUZIONE (PPPL per-claim) ───────────────
    pppl_claim_raw  = claim.get("pppl_claim")
    pppl_claim_norm = _normalize_pppl(
        pppl_claim_raw,
        n_tokens=claim.get("pppl_claim_tokens"),
        pppl_min=1.66,
        pppl_max=160
    )
    R_translation   = pppl_claim_norm


    # ── Combinazione lineare ──────────────────────────────────
    risk = alpha * R_transcript + (1 - alpha) * R_translation
    # Evita che una PPPL bassa renda il sistema troppo ottimista
    risk = max(risk, 0.85 * R_transcript)

    if risk < THRESHOLDS["green"]:
        risk_level = "green"
    elif risk < THRESHOLDS["yellow"]:
        risk_level = "yellow"
    else:
        risk_level = "red"

    # Contributi per diagnostica (decomposizione del risk finale)
    contributions_transcript = {
    "conf_min_worst":  round((1 - sig["conf_min_worst"]) * w_transcript["conf_min_worst"], 4),
    "conf_mean":       round((1 - sig["conf_mean"]) * w_transcript["conf_mean"], 4),
    "conf_geo_period": round((1 - conf_geo) * w_transcript["conf_geo_period"], 4),
    }
    top_transcript = max(contributions_transcript, key=contributions_transcript.get)

    return {
        "risk_score":              risk,
        "risk_level":              risk_level,
        "alpha":                   round(alpha, 2),
        "R_transcript":            round(R_transcript, 4),
        "R_translation":           round(R_translation, 4),
        "contributions_transcript": contributions_transcript,
        "top_contributor_transcript": top_transcript,
        "pppl_claim_raw":          pppl_claim_raw,
        "pppl_claim_norm":         round(pppl_claim_norm, 4),
    }


def score_all_claims(validated_claims: list) -> list:
    """
    Applica compute_risk_score a ogni claim.
    La firma NON riceve più risultato_ensemble: la PPPL è già
    per-claim dentro claim["pppl_claim"] iniettata allo STEP 6.1.
    """
    scored = []
    for claim in validated_claims:
        risk_data = compute_risk_score(claim)
        scored.append({**claim, **risk_data})
    return scored


def compute_overall_risk(scored_claims: list) -> dict:
    """
    Calcola l'overall risk a partire dai colori dei claim,
    pesati in base al claim_type.

    Restituisce anche una spiegazione testuale della regola applicata.
    """

    if not scored_claims:
        return {
            "overall_risk_level": "green",
            "overall_reason": "Nessun claim presente: rischio complessivo impostato a green."
        }

    claim_type_weights = {
        "richiesta_farmaco":  1.5,
        "sintomo":            1.5,
        "condizione_medica":  1.5,
        "azione":             1.0,
        "informazione":       0.8,
        "domanda_generica":   0.7,
    }

    critical_types = {
        "richiesta_farmaco",
        "sintomo",
        "condizione_medica",
    }

    total_weight = 0.0
    red_weight = 0.0
    yellow_weight = 0.0

    n_red = 0
    n_yellow = 0
    n_green = 0

    has_critical_red = False
    critical_red_types_found = []

    for claim in scored_claims:
        claim_type = claim.get("claim_type", "informazione")
        risk_level = claim.get("risk_level", "green")

        w = claim_type_weights.get(claim_type, 1.0)
        total_weight += w

        if risk_level == "red":
            red_weight += w
            n_red += 1

            if claim_type in critical_types:
                has_critical_red = True
                critical_red_types_found.append(claim_type)

        elif risk_level == "yellow":
            yellow_weight += w
            n_yellow += 1

        elif risk_level == "green":
            n_green += 1

    if total_weight == 0:
        return {
            "overall_risk_level": "green",
            "overall_reason": "Peso totale nullo: rischio complessivo impostato a green."
        }

    p_red = red_weight / total_weight
    p_yellow = yellow_weight / total_weight
    p_risky = (red_weight + yellow_weight) / total_weight

    n_claims = len(scored_claims)

    if has_critical_red:
        overall_level = "red"
        reason = (
            "Overall impostato a red perché è presente almeno un claim red "
            f"di tipo clinicamente critico: {sorted(set(critical_red_types_found))}."
        )

    elif n_claims <= 3 and n_red >= 1:
        overall_level = "red"
        reason = (
            "Overall impostato a red perché il numero di claim è basso "
            f"({n_claims}) ed è presente almeno un claim red."
        )

    elif p_red >= 0.20:
        overall_level = "red"
        reason = (
            "Overall impostato a red perché la quota pesata di claim red "
            f"è {p_red:.2%}, quindi supera la soglia del 20%."
        )

    elif p_risky >= 0.40:
        overall_level = "yellow"
        reason = (
            "Overall impostato a yellow perché la quota pesata di claim non-green "
            f"(yellow + red) è {p_risky:.2%}, quindi supera la soglia del 40%."
        )

    elif p_yellow >= 0.25:
        overall_level = "yellow"
        reason = (
            "Overall impostato a yellow perché la quota pesata di claim yellow "
            f"è {p_yellow:.2%}, quindi supera la soglia del 25%."
        )

    else:
        overall_level = "green"
        reason = (
            "Overall impostato a green perché non sono presenti condizioni sufficienti "
            "per classificare il rischio complessivo come yellow o red."
        )

    return {
        "overall_risk_level": overall_level,
        "overall_reason": reason,
    }


# Cella 16 - Esecuzione Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE COMPLETA — esegue tutta la catena per ogni audio
# e produce per ciascun file due report (.json + .txt) in
# /content/reports/<nome_audio_senza_ext>/
# ═══════════════════════════════════════════════════════════════


SEP = "=" * 60

class _ReportLog:
    def __init__(self):
        self.lines = []
    def __call__(self, *args, sep=" ", end="\n"):
        line = sep.join(str(a) for a in args)
        print(line, end=end)
        self.lines.append(line + ("" if end == "\n" else end))
    def section(self, title):
        bar = "=" * 70
        self(bar)
        self(title)
        self(bar)
    def text(self):
        return "".join(self.lines) if any(l.endswith("\n") for l in self.lines) \
               else "\n".join(self.lines)


# ── Loop principale su tutti i file audio caricati ───────────
all_reports_summary = []

for _audio_idx, (PERCORSO_AUDIO, recording_doc) in enumerate(
        zip(audio_files, audio_docs), start=1):
    log = _ReportLog()
    log.section(f"AUDIO {_audio_idx}/{len(audio_files)}: {os.path.basename(PERCORSO_AUDIO)}")

    report = {
        "audio_file":     os.path.basename(PERCORSO_AUDIO),
        "audio_path":     PERCORSO_AUDIO,
        "timestamp":      _now_ms(),
        "pipeline_steps": {},
        "errors":         [],
    }



    # Variabili locali per raccogliere i risultati di ogni step
    # prima di qualsiasi scrittura su MongoDB
    risultato_rag       = None
    dati_analisi_full   = None
    output_fonetico     = None
    risultato_ensemble  = None
    pppl_result         = None   # PPPL sul testo italiano normalizzato
    claim_list          = None
    validated_claims    = None
    scored_claims       = None
    overall             = None


    session_id = None  # ← inizializza prima del try

    try:
        session_id = init_new_session(recording_doc=recording_doc)

        # ════════════════════════════════════════════════════════
        # STEP 1 — TRASCRIZIONE WHISPER + RAG
        # ════════════════════════════════════════════════════════
        t_s1 = _now_ms()
        audio, sr = librosa.load(PERCORSO_AUDIO, sr=16_000, mono=True)
        risultato_rag = pipeline_rag.trascrivi(audio, verbose=False)
        t_e1 = _now_ms()

        log("")
        log("=" * 55)
        log("RIEPILOGO WHISPER RAG")
        log("=" * 55)
        log(f"Sporco  (prima passata):   {risultato_rag['testo_sporco']}")
        log(f"Finale  (seconda passata): {risultato_rag['testo_finale']}")
        log("")
        log("Frasi napoletane usate per il prompt:")
        for i, f in enumerate(risultato_rag['frasi_recuperate']):
            log(f"  [{i+1}] score={f['score']:.3f} | {f['napoletano']}")
        log("")
        log(f"Prompt  (usato):           {risultato_rag['prompt_usato']}")
        log("")

        report["pipeline_steps"]["whisper_rag"] = {
            "testo_sporco":       risultato_rag.get("testo_sporco"),
            "testo_finale":       risultato_rag.get("testo_finale"),
            "prompt_usato":       risultato_rag.get("prompt_usato"),
            "frasi_recuperate":   risultato_rag.get("frasi_recuperate"),
        }

        # ════════════════════════════════════════════════════════
        # STEP 2 — ANALISI CONFIDENZA TOKEN/PAROLA
        # ════════════════════════════════════════════════════════
        result            = risultato_rag["result"]
        testo_per_analisi = risultato_rag["testo_finale"]

        testo_sporco = risultato_rag["testo_sporco"]

        log("")
        log("=" * 60)
        log("TRASCRIZIONE:")
        log(testo_per_analisi)
        log("=" * 60)
        log(f"Confidence media aritmetica : {result['period_conf_mean']:.4f}")
        log(f"Confidence media geometrica : {result['period_conf_geo']:.4f}")
        log("=" * 60)

        df_tokens = pd.DataFrame(result["tokens"])
        log("")
        log("CONFIDENCE PER TOKEN (prime 30):")
        log(df_tokens[["token_text", "confidence"]].head(30).to_string(index=False))

        THRESHOLD = 0.70
        low_conf  = df_tokens[df_tokens["confidence"] < THRESHOLD]
        log("")
        log(f"Token con confidence < {THRESHOLD}:")
        log(low_conf[["token_text", "confidence"]].to_string(index=False))

        t_s2 = _now_ms()
        word_data  = tokens_to_words(result["tokens"])
        t_e2 = _now_ms()

        df_words   = pd.DataFrame(word_data)

        log("")
        log("CONFIDENCE PER PAROLA:")
        log(df_words[["word", "n_tokens", "conf_mean", "conf_min", "conf_geo"]].to_string(index=False))

        low_conf_words = df_words[df_words["conf_mean"] < THRESHOLD]
        log("")
        log(f"Parole con conf_mean < {THRESHOLD}:")
        log(low_conf_words[["word", "conf_mean", "conf_min", "tokens"]].to_string(index=False))

        report["pipeline_steps"]["confidence_analysis"] = {
            "trascrizione":     testo_per_analisi,
            "period_conf_mean": float(result["period_conf_mean"]),
            "period_conf_geo":  float(result["period_conf_geo"]),
            "tokens":           result["tokens"],
            "words":            word_data,
            "threshold_used":   THRESHOLD,
            "low_conf_tokens":  low_conf.to_dict(orient="records"),
            "low_conf_words":   low_conf_words.to_dict(orient="records"),
        }

        dati_analisi_full = {
            "period_conf_mean": result["period_conf_mean"],
            "period_conf_geo":  result["period_conf_geo"],
            "threshold_used":   THRESHOLD,
            "tokens":           result["tokens"],
            "words":            word_data,
            "low_conf_tokens":  low_conf.to_dict(orient="records"),
            "low_conf_words":   low_conf_words.to_dict(orient="records"),
        }

        # ════════════════════════════════════════════════════════
        # STEP 3 — TRASCRIZIONE FONETICA (PhoneticXeus)
        # ════════════════════════════════════════════════════════
        t_s3 = _now_ms()
        output_fonetico = transcribe_with_phonetic_xeus_fixed(PERCORSO_AUDIO, inference)
        t_e3 = _now_ms()

        log("")
        log("--- RISULTATO FONETICO ---")
        log(f"Trascrizione IPA: {output_fonetico}")
        log("--------------------------")

        report["pipeline_steps"]["phonetic_xeus"] = {
            "trascrizione_ipa": output_fonetico,
        }

        # ════════════════════════════════════════════════════════
        # STEP 4+5 — ANALISI SEMANTICA LLM
        # Versione senza analisi sintattica parola-per-parola
        # ════════════════════════════════════════════════════════

        t_s4 = _now_ms()

        risultato_ensemble = analyze_and_normalize_with_llm(
            transcript_whisper_rag          = testo_per_analisi,
            transcrpit_whisper_testo_sporco = testo_sporco,
            transcript_PhoneticXeus         = output_fonetico,
            word_data                       = word_data,
        )

        t_e5 = _now_ms()

        log("")
        log("ANALISI SEMANTICA LLM:")
        log("  Analisi sintattica parola-per-parola disattivata.")

        report["pipeline_steps"]["llm_semantic_analysis"] = {
            "syntactic_analysis_enabled": False,
        }

        log("")
        log("=" * 60)
        log("🎯 RISULTATO ENSEMBLE (Whisper-RAG + PhoneticXeus + LLM):")
        log("=" * 60)
        log(f"Trascrizione Whisper-RAG  : {testo_per_analisi}")
        log(f"Trascrizione PhoneticXeus : {output_fonetico}")
        log("-" * 60)
        log(f"Italiano Standard    : {risultato_ensemble.get('normalized_text', 'Dato mancante')}")
        log(f"Dominio              : {risultato_ensemble.get('domain', 'Non specificato')}")
        log("")
        log("Problemi Risolti/Rilevati:")
        issues = risultato_ensemble.get("semantic_issues", [])
        if not issues:
            log("  Nessun problema rilevato o lista mancante.")
        else:
            for issue in issues:
                words = issue.get("words", [issue.get("word", "Sconosciuta")])
                indices = issue.get("word_indices", [issue.get("word_index", "Indice mancante")])
                reason = issue.get("reason", "Nessuna spiegazione fornita")

                words_str = ", ".join(words)
                indices_str = ", ".join(map(str, indices))

                log(f"  - [{words_str}] indici={indices_str} → {reason}")

        report["pipeline_steps"]["ensemble_llm"] = risultato_ensemble

        # timestamp per ensemble_llm (STEP 4+5 unificato)
        t_s5 = t_s4
        t_e4 = t_e5

        # ════════════════════════════════════════════════════════
        # STEP 5.1 — PSEUDO-PERPLEXITY DI SESSIONE (testo normalizzato)
        # Calcolata sul normalized_text prodotto dall'LLM (italiano standard).
        # Segnale globale di sessione: misura la fluenza complessiva post-LLM.
        # Salvata in pipeline_stages come "5.1_pppl_session".
        # ════════════════════════════════════════════════════════
        testo_normalizzato_it = risultato_ensemble.get("normalized_text", "")

        t_s51 = _now_ms()
        pppl_session_result = compute_pppl(testo_normalizzato_it, pppl_min=1.98, pppl_max=30)
        t_e51 = _now_ms()

        log("")
        log("📊 STEP 5.1 — PPPL SESSIONE (testo italiano normalizzato)")
        log(f"   Testo analizzato : {testo_normalizzato_it}")
        log(f"   PPPL             : {pppl_session_result['pppl']}  (più bassa = più fluente)")
        log(f"   log2(PPPL)       : {pppl_session_result['log_pppl']}")
        log(f"   Token analizzati : {pppl_session_result['n_tokens']}")
        log(f"   pppl_norm        : {pppl_session_result['pppl_norm']}")
        log(f"   Mean log P       : {pppl_session_result['mean_log_prob']}")
        log(f"   Modello BERT     : {pppl_session_result['model']}")
        log(f"   Tempo (ms)       : {t_e51 - t_s51}")


        # ════════════════════════════════════════════════════════
        # STEP 6 — CLAIM SEGMENTATION + VALIDAZIONE LLM
        # ════════════════════════════════════════════════════════

        t_s6 = _now_ms()

        claim_list = extract_spans_from_spacy(risultato_ensemble["normalized_text"])

        t_e6 = _now_ms()

        log("")
        log("=" * 60)
        log("CLAIM LIST ESTRATTA CON SPACY")
        log("=" * 60)

        for i, claim in enumerate(claim_list):
            log(f"\n📌 Claim {i+1}: \"{claim['claim_text']}\"")

        report["pipeline_steps"]["claim_extraction"] = {
            "claim_list": claim_list,
        }

        # ════════════════════════════════════════════════════════
        # STEP 7 — VALIDAZIONE LLM + ALLINEAMENTO PAROLE + SEGNALI ASR
        # ════════════════════════════════════════════════════════

        t_s7 = _now_ms()

        validated_claims = validate_claims_with_llm(
            claim_list=claim_list,
            normalized_text=risultato_ensemble["normalized_text"],
            word_data=word_data,
        )

        t_e7 = _now_ms()

        # ════════════════════════════════════════════════════════
        # STEP 7.1 — PPPL PER-CLAIM (sul claim_text)
        # È il segnale dell'aliquota TRADUZIONE nel risk score.
        # I risultati vengono iniettati direttamente in ogni claim validato.
        # ════════════════════════════════════════════════════════

        t_s71 = _now_ms()

        log("")
        log("=" * 60)
        log("STEP 7.1 — PPPL PER-CLAIM (aliquota traduzione)")
        log("=" * 60)

        for vc in validated_claims:

            text_for_pppl = vc.get("claim_text", "")
            pppl_c = compute_pppl(text_for_pppl, pppl_min=1.66, pppl_max=160)
            vc["pppl_claim"]        = pppl_c.get("pppl")
            vc["pppl_claim_log"]    = pppl_c.get("log_pppl")
            vc["pppl_claim_tokens"] = pppl_c.get("n_tokens")
            log(f"   📌 [{vc['claim_text'][:50]}] → pppl_claim={pppl_c.get('pppl')} "
                f"(n_tok={pppl_c.get('n_tokens')})")

        t_e71 = _now_ms()
        log(f"   Tempo totale PPPL per-claim (ms): {t_e71 - t_s71}")

        log("")
        log("=" * 60)
        log("CLAIM LIST VALIDATA DALL'LLM CON PAROLE ALLINEATE")
        log("=" * 60)

        for i, claim in enumerate(validated_claims):
            sig = claim["asr_signals"]

            log(f"\n📌 Claim {i+1}: \"{claim['claim_text']}\"")
            log(f"   Tipo               : {claim['claim_type']}")
            log(f"   È una domanda?     : {claim['is_question']}")
            log(f"   Ambiguo?           : {claim['ambiguous']}")

            if claim.get("ambiguity_reason"):
                log(f"   Motivo ambiguità   : {claim['ambiguity_reason']}")

            log(f"   Indici parole RAG  : {claim['source_word_indices']}")
            log(f"   Parole originali   : {claim['source_words']}")
            log(f"   conf_min_worst     : {sig['conf_min_worst']}")
            log(f"   conf_mean          : {sig['conf_mean']}")

        report["pipeline_steps"]["claim_validation"] = {
            "validated_claims": validated_claims,
        }

        # ════════════════════════════════════════════════════════
        # STEP 8 — RISK SCORING
        # ════════════════════════════════════════════════════════

        t_s8 = _now_ms()
        scored_claims = score_all_claims(validated_claims)
        overall       = compute_overall_risk(scored_claims)
        t_e8 = _now_ms()

        EMOJI = {"green": "🟢", "yellow": "🟡", "red": "🔴"}

        log("")
        log("=" * 60)
        log("RISK SCORING PER CLAIM")
        log("=" * 60)
        for i, claim in enumerate(scored_claims):
            emoji = EMOJI[claim["risk_level"]]
            log(f"\n{emoji} Claim {i+1}: \"{claim['claim_text']}\"")
            log(f"   Tipo              : {claim['claim_type']}")
            log(f"   Risk score        : {claim['risk_score']}")
            log(f"   Risk level        : {claim['risk_level'].upper()}")
            log(f"   α (transcript)    : {claim['alpha']}")
            log(f"   R_transcript      : {claim['R_transcript']}")
            log(f"   R_translation     : {claim['R_translation']}")
            log(f"   PPPL claim raw    : {claim['pppl_claim_raw']}")
            log(f"   PPPL claim norm   : {claim['pppl_claim_norm']}")
            log(f"   Top contributor   : {claim['top_contributor_transcript']}")
            log(f"   Contributi ASR    : {claim['contributions_transcript']}")

        log("")
        log("=" * 60)
        emoji_overall = EMOJI[overall["overall_risk_level"]]
        log(f"{emoji_overall} OVERALL RISK: "
            f"{overall['overall_risk_level'].upper()}")
        log(
            f"REASON: {overall['overall_reason']}")
        log("=" * 60)

        report["pipeline_steps"]["risk_scoring"] = {
            "scored_claims": scored_claims,
            "overall":       overall,
        }

        all_reports_summary.append({
            "audio_file":         report["audio_file"],
            "normalized_text":    risultato_ensemble.get("normalized_text"),
            "overall_risk_level": overall["overall_risk_level"],
            "n_claims":           len(scored_claims),
        })


        # ════════════════════════════════════════════════════════
        # STEP 9 — METRICHE VALUTATIVE PIPE
        # ════════════════════════════════════════════════════════

        prompt_text_originale = recording_doc.get("promptText", "")
        testo_tradotto_llm    = risultato_ensemble.get("normalized_text", "")

        translation_metrics = {
            "cosine_similarity": None,
            "bleu":              None,
            "rouge1":            None,
            "rouge2":            None,
            "rougeL":            None,
            "wer":               None,
            "prompt_text":       prompt_text_originale,
            "translated_text":   testo_tradotto_llm,
            "embedding_model":   CONFIG["embedding_model"],
        }

        t_s9 = _now_ms()
        if prompt_text_originale and testo_tradotto_llm:

            # ── 1. COSENO SIMILARITY ─────────────────────────────
            # Semantica profonda — gestisce parafrasi e variazioni dialettali
            # Valore: 0-1, più alto = più simile
            embedder      = pipeline_rag.retriever.embedder
            emb_originale = embedder.encode(
                prompt_text_originale,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            emb_tradotto = embedder.encode(
                testo_tradotto_llm,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            cosine_score = round(float(np.dot(emb_originale, emb_tradotto)), 4)
            translation_metrics["cosine_similarity"] = cosine_score

            # ── 2. BLEU ──────────────────────────────────────────
            # Misura sovrapposizione di n-grammi tra testo prodotto e riferimento
            # Pensato per machine translation — penalizza variazioni lessicali
            # Valore: 0-100, più alto = più simile
            # NOTA: su testi brevi tende a dare score bassi — usare come segnale
            # relativo tra recording, non come valore assoluto
            bleu_metric = BLEU(effective_order=True)  # effective_order=True gestisce testi brevi
            bleu_result = bleu_metric.sentence_score(
                hypothesis=testo_tradotto_llm,
                references=[prompt_text_originale],
            )
            translation_metrics["bleu"] = round(bleu_result.score, 4)

            # ── 3. ROUGE ─────────────────────────────────────────
            # Misura recall di n-grammi (quanto del testo originale è coperto)
            # ROUGE-1: unigrammi, ROUGE-2: bigrammi, ROUGE-L: sottosequenza comune
            # Valore: 0-1 (precision, recall, F1) — usiamo F1
            # Più robusto di BLEU su testi brevi
            scorer_rouge = rouge_scorer.RougeScorer(
                ["rouge1", "rouge2", "rougeL"],
                use_stemmer=False   # False: italiano non ha stemmer integrato
            )
            rouge_scores = scorer_rouge.score(
                target=prompt_text_originale,
                prediction=testo_tradotto_llm,
            )
            translation_metrics["rouge1"] = round(rouge_scores["rouge1"].fmeasure, 4)
            translation_metrics["rouge2"] = round(rouge_scores["rouge2"].fmeasure, 4)
            translation_metrics["rougeL"] = round(rouge_scores["rougeL"].fmeasure, 4)

            # ── 4. WER (Word Error Rate) ─────────────────────────
            # Misura quante parole differiscono (sostituzioni + inserzioni + cancellazioni)
            # Pensato per ASR — qui lo usiamo per quantificare la distanza lessicale
            # Valore: 0-1+ (può superare 1 se ci sono molte inserzioni)
            # Più basso = più simile; 0 = identici
            wer_score = round(wer(
                reference=prompt_text_originale,
                hypothesis=testo_tradotto_llm,
            ), 4)
            translation_metrics["wer"] = wer_score


            log("")
            log("📐 TRANSLATION QUALITY METRICS")
            log(f"   Cosine Similarity : {cosine_score:.4f}  (0-1,  più alto = meglio)")
            log(f"   BLEU              : {translation_metrics['bleu']:.4f}  (0-100, più alto = meglio)")
            log(f"   ROUGE-1           : {translation_metrics['rouge1']:.4f}  (0-1,  più alto = meglio)")
            log(f"   ROUGE-2           : {translation_metrics['rouge2']:.4f}  (0-1,  più alto = meglio)")
            log(f"   ROUGE-L           : {translation_metrics['rougeL']:.4f}  (0-1,  più alto = meglio)")
            log(f"   WER               : {wer_score:.4f}  (0-1+, più basso = meglio)")

        else:
            log("⚠️  Metriche non calcolate — promptText o traduzione mancanti")

        t_e9 = _now_ms()

        # ════════════════════════════════════════════════════════
        # ✅ SCRITTURA SU MONGODB — solo qui, solo se tutti gli
        #    step precedenti sono andati a buon fine senza errori
        # ════════════════════════════════════════════════════════
        log("")
        log("=" * 60)
        log("SALVATAGGIO SU MONGODB")
        log("=" * 60)

        save_transcript_output(session_id, risultato_rag, dati_analisi_full, t_start=t_s1, t_end=t_e2)
        log("  ✅ save_transcript_output")

        save_pipeline_stage(session_id, "3_phonetic_xeus", {"ipa_text": output_fonetico}, t_start=t_s3, t_end=t_e3)
        log("  ✅ save_pipeline_stage: 3_phonetic_xeus")

        save_pipeline_stage(
            session_id,
            "4_llm_semantic_analysis",
            {
                "syntactic_analysis_enabled": False,
            },
        t_start=t_s4,
        t_end=t_e4,
        )
        log("  ✅ save_pipeline_stage: 4_llm_semantic_analysis")

        save_pipeline_stage(session_id, "5_ensemble_llm", risultato_ensemble, t_start=t_s5, t_end=t_e5)
        log("  ✅ save_pipeline_stage: 5_ensemble_llm")

        # ── STEP 5.1: salvataggio PPPL sessione ──────────────────
        save_pipeline_stage(session_id, "5_1_pppl_session", pppl_session_result, t_start=t_s51, t_end=t_e51)
        log("  ✅ save_pipeline_stage: 5_1_pppl_session")

        save_pipeline_stage(session_id, "6_claim_extraction", claim_list, t_start=t_s6, t_end=t_e6)
        log("  ✅ save_pipeline_stage: 6_claim_extraction")

        save_risk_scoring(
            session_id,
            risk_scores_output  = copy.deepcopy(scored_claims),
            t_start             = t_s8,
            t_end               = t_e8,
            t_validation_start  = t_s7,
            t_validation_end    = t_e7,
            t_pppl_start        = t_s71,
            t_pppl_end          = t_e71,
        )
        log("  ✅ save_risk_scoring")


        save_translation_metrics(session_id, translation_metrics, t_start=t_s9, t_end=t_e9)
        log("  ✅ save_translation_metrics")

        complete_session(session_id, success=True, overall=overall)
        log("  ✅ complete_session → status: completed")

    except Exception as e:
        err = traceback.format_exc()
        log("")
        log("!" * 60)
        log(f"ERRORE durante l'elaborazione di {PERCORSO_AUDIO}:")
        log(err)
        log("!" * 60)
        report["errors"].append({
            "message":   str(e),
            "traceback": err,
        })

        if session_id is not None:          # ← solo se la sessione era stata creata
            complete_session(session_id, success=False, error_msg=str(e))
            log("  ⚠️  MongoDB: sessione marcata come failed")
        else:
            log("  ⚠️  Sessione non inizializzata — nessun dato scritto su MongoDB")

    # ════════════════════════════════════════════════════════════
    # SALVATAGGIO REPORT LOCALE (sempre, anche in caso di errore)
    # ════════════════════════════════════════════════════════════
    stem      = Path(PERCORSO_AUDIO).stem.replace(" ", "_")
    audio_dir = REPORTS_DIR / stem
    audio_dir.mkdir(parents=True, exist_ok=True)

    json_path = audio_dir / f"{stem}__report.json"
    txt_path  = audio_dir / f"{stem}__report.txt"

    with open(json_path, "w", encoding="utf-8") as f:
        _json.dump(report, f, ensure_ascii=False, indent=2, default=_json_default)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(log.text())

    print(f"\n💾 Report salvati in:")
    print(f"   {json_path}")
    print(f"   {txt_path}")


# ─── Salvataggio del riepilogo cross-audio ────────────────────
summary_path = REPORTS_DIR / "ALL_AUDIO_SUMMARY.json"
with open(summary_path, "w", encoding="utf-8") as f:
    _json.dump(all_reports_summary, f, ensure_ascii=False, indent=2, default=_json_default)

print(f"\n📊 Riepilogo globale di {len(all_reports_summary)} audio salvato in:")
print(f"   {summary_path}")
print("\n✅ Pipeline completata su tutti i file.")

Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AUDIO 1/50: prompt-10189_rec-14.wav


Both `max_new_tokens` (=171) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RIEPILOGO WHISPER RAG
Sporco  (prima passata):   A tog un generico dell'ultracet, a volte al tog anche a quattro volte al dì.
Finale  (seconda passata): a tog un generico de l'ultracet a volti al tog anche quattro volti al di

Frasi napoletane usate per il prompt:
  [1] score=0.633 | ma si invec e m' piglià o sciropp pa' toss fluifort m pigl o flubason è a stessa cos?
  [2] score=0.622 | ma l'antibiotic s adda piglia a matin o a ser? e aggia pur magna primm ro piglia?
  [3] score=0.622 | Buongiorno dotto', a nu poc i tiemp, vac spiss 'rind u bagn 'teng i rin' lasch
  [4] score=0.620 | quann er piccerill m operai e tonsill, me l'hann luat

Prompt  (usato):           ma si invec e m' piglià o sciropp pa' toss fluifort m pigl o flubason è a stessa cos? ma l'antibiotic s adda piglia a matin o a ser? e aggia pur magna primm ro piglia? Buongiorno dotto', a nu poc i tiemp, vac spiss 'rind u bagn 'teng i rin' lasch quann er piccerill m operai e tonsill, me l'hann luat 'nu poc tutt doie chin n

Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


RIEPILOGO WHISPER RAG
Sporco  (prima passata):   Un resto a me denodumurasi
Finale  (seconda passata): onore suonami dei nodi morassi

Frasi napoletane usate per il prompt:
  [1] score=0.830 | Ca maronn t’accumpagn
  [2] score=0.796 | Dicette o pappice vicin a noce, damme o tiempo ca te spertose
  [3] score=0.783 | me fatt scennr e uallr nderr
  [4] score=0.770 | Accussì s car malat

Prompt  (usato):           Ca maronn t’accumpagn Dicette o pappice vicin a noce, damme o tiempo ca te spertose me fatt scennr e uallr nderr Accussì s car malat 'nu poc tutt doie chin nguoll spiss spess appicciat applat scassat ultimament 'o 'a 'e nu na ll' si' mo' ca pe' 'ncopp 'mmiezo teng stong sent sento vac fann bruc pigliat acal m'avot ammsuratam vulit scriver fa mal fann mal m fa mal m sent m bruc tengo tene tenimmo pozzo po' potimmo stongo sta stammo aggio ha avimmo vaco va iammo saccio sa sapimmo voglio vo' vulimmo dico dice dicimmo faccio fa facimmo


TRASCRIZIONE:
onore suonami dei nodi morassi


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ save_pipeline_stage: 5_1_pppl_session
  ✅ save_pipeline_stage: 6_claim_extraction
  ✅ save_risk_scoring
  ✅ save_translation_metrics
  ✅ complete_session → status: completed

💾 Report salvati in:
   /kaggle/working/reports/prompt-9602_rec-9/prompt-9602_rec-9__report.json
   /kaggle/working/reports/prompt-9602_rec-9/prompt-9602_rec-9__report.txt
AUDIO 50/50: prompt-12429_rec-12.wav


Both `max_new_tokens` (=229) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RIEPILOGO WHISPER RAG
Sporco  (prima passata):   oggi è tenuto il problema e ritornare in passato
Finale  (seconda passata): Aggi tenuto il problema e ritorno di vino. Passato.

Frasi napoletane usate per il prompt:
  [1] score=0.786 | Fa ben e scuordt, fa mal e pienzace
  [2] score=0.734 | Jamme, facimmo 'e ccose buone e nun ce pensa'.
  [3] score=0.723 | Dicette o pappice vicin a noce, damme o tiempo ca te spertose
  [4] score=0.714 | Accussì s car malat

Prompt  (usato):           Fa ben e scuordt, fa mal e pienzace Jamme, facimmo 'e ccose buone e nun ce pensa'. Dicette o pappice vicin a noce, damme o tiempo ca te spertose Accussì s car malat 'nu poc tutt doie chin nguoll spiss spess appicciat applat scassat ultimament 'o 'a 'e nu na ll' si' mo' ca pe' 'ncopp 'mmiezo teng stong sent sento vac fann bruc pigliat acal m'avot ammsuratam vulit scriver fa mal fann mal m fa mal m sent m bruc tengo tene tenimmo pozzo po' potimmo stongo sta stammo aggio ha avimmo vaco va iammo saccio sa sap

In [ ]:
import shutil
import os
from pathlib import Path

# ── CONFIGURAZIONE PERCORSI ────────────────────────────────
# Su Kaggle i report creati nei passi precedenti saranno in /kaggle/working/reports
CARTELLA_DA_SCARICARE = "/kaggle/working/reports"
NOME_ZIP = "download_reports"
# ───────────────────────────────────────────────────────────

# Definiamo il percorso di output nella cartella working di Kaggle
output_zip_path = f"/kaggle/working/{NOME_ZIP}"

# Verifica se la cartella esiste prima di zippare
if os.path.exists(CARTELLA_DA_SCARICARE):
    # Crea l'archivio .zip
    # shutil.make_archive aggiungerà automaticamente l'estensione .zip
    shutil.make_archive(output_zip_path, "zip", CARTELLA_DA_SCARICARE)
    print(f"✅ Archivio creato correttamente: {output_zip_path}.zip")
    print(f"📂 Puoi scaricarlo dal pannello 'Data' -> 'Output' a destra.")
else:
    print(f"❌ Errore: La cartella {CARTELLA_DA_SCARICARE} non esiste.")

In [ ]:
# %% [code]
import os
import uuid
import time
import shutil
import librosa
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import threading
import copy

# 1. Recupero dinamico e sicuro del database già esistente nel Notebook
if 'db' in globals():
    # Se l'oggetto database globale 'db' esiste già, usiamo direttamente quello
    pass
elif 'client' in globals():
    # Se esiste il client ma non l'oggetto db, lo associamo usando il nome del tuo DB
    _db_name = globals().get('DB_NAME', 'asr_dialects_no_analisi_sintattica')
    db = client[_db_name]
else:
    raise RuntimeError(
        "❌ ERRORE: Impossibile trovare una connessione a MongoDB attiva!\n"
        "Assicurati di aver eseguito tutte le celle superiori del notebook "
        "che contengono l'inizializzazione di 'client' o 'db' prima di lanciare il server."
    )

app = FastAPI(title="ASR Dialetti - Realtime Inference Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def _now_ms() -> int:
    return int(time.time() * 1000)

# Helper locale per la chiusura e l'aggiornamento dello stato della sessione su MongoDB
def _local_complete_session(session_id: str, success: bool, error_msg: str = None, overall_risk = None):
    status_str = "completed" if success else "failed"
    update_data = {
        "status": status_str,
        "completed_at": _now_ms()
    }
    if error_msg:
        update_data["error_log"] = error_msg
    if overall_risk:
        update_data["overall_risk"] = overall_risk

    db["Sessions"].update_one({"_id": session_id}, {"$set": update_data})

@app.post("/process_audio")
async def process_audio(
    audio: UploadFile = File(...),
    ageRange: str = Form(...),
    gender: str = Form(...),
    dialect: str = Form(...),
    livingContext: str = Form(...),
    filename: str = Form(...)
):
    t_inizio_pipeline = _now_ms()
    session_id = str(uuid.uuid4())

    os.makedirs("/kaggle/working/temp_audios", exist_ok=True)
    local_audio_path = os.path.join("/kaggle/working/temp_audios", f"{session_id}_{filename}")

    with open(local_audio_path, "wb") as buffer:
        shutil.copyfileobj(audio.file, buffer)

    try:
        print(f"\n🚀 Avvio elaborazione realtime per: {filename}")

        # 2. Inserimento Profilo Paziente
        participant_doc = {
            "_id": f"pax_{session_id[:8]}",
            "gender": gender,
            "ageRange": ageRange,
            "dialect": dialect,
            "education": "Non Specificato",
            "livingContext": livingContext,
            "dialectFrequency": "Spesso"
        }
        db['participants'].insert_one(participant_doc)

        # Inserimento record di Sessione iniziale
        session_doc = {
            "_id": session_id,
            "participant_id": participant_doc["_id"],
            "filename": filename,
            "promptCategory": "realtime_upload",
            "started_at": t_inizio_pipeline,
            "status": "processing"
        }
        db['Sessions'].insert_one(session_doc)

        # 3. Caricamento Audio Array (16kHz)
        audio_array, sr = librosa.load(local_audio_path, sr=16000)

        # ════════════════════════════════════════════════════════
        # STEP 1 & 2 — WHISPER RAG & ANALISI CONFIDENZA
        # ════════════════════════════════════════════════════════
        t_s1 = _now_ms()
        rag_output = pipeline_rag.trascrivi(audio_array, verbose=False)
        t_e1 = _now_ms()

        testo_sporco = rag_output["testo_sporco"]
        testo_finale = rag_output["testo_finale"]
        prompt_usato = rag_output.get("prompt_usato", "")
        frasi_recuperate = rag_output.get("frasi_recuperate", [])

        result = rag_output["result"]
        THRESHOLD = 0.70
        word_data = tokens_to_words(result["tokens"])

        analysis_results = {
            "period_conf_mean": float(result.get("period_conf_mean", 0)),
            "period_conf_geo": float(result.get("period_conf_geo", 0)),
            "threshold_used": THRESHOLD,
            "tokens": result.get("tokens", []),
            "words": word_data,
            "low_conf_tokens": [t for t in result.get("tokens", []) if t["confidence"] < THRESHOLD],
            "low_conf_words": [w for w in word_data if w["conf_mean"] < THRESHOLD]
        }

        db["Transcripts"].replace_one(
            {"_id": session_id},
            {
                "_id": session_id,
                "started_at": t_s1,
                "saved_at": t_e1,
                "rag_stage": {
                    "testo_sporco": testo_sporco,
                    "testo_finale": testo_finale,
                    "prompt_usato": prompt_usato,
                    "frasi_recuperate": frasi_recuperate
                },
                "analysis_stage": analysis_results
            },
            upsert=True
        )

        # ════════════════════════════════════════════════════════
        # STEP 3 — RICONOSCIMENTO FONETICO IPA
        # ════════════════════════════════════════════════════════
        t_s3 = _now_ms()
        try:
            ipa_transcript = transcribe_with_phonetic_xeus_fixed(local_audio_path, inference)
        except Exception as e:
            print(f"Errore PhoneticXeus: {e}")
            ipa_transcript = "[Errore: Modello Fonetico non caricato]"
        t_e3 = _now_ms()

        db["Pipeline_Stages"].update_one(
            {"_id": session_id},
            {"$set": {"stages.3_phonetic_xeus": {"data": {"ipa_text": ipa_transcript}, "started_at": t_s3, "saved_at": t_e3}}},
            upsert=True
        )

        # ════════════════════════════════════════════════════════
        # STEP 4+5 — ANALISI SEMANTICA (Ensemble LLM) - No Sintattica
        # ════════════════════════════════════════════════════════
        t_s4 = _now_ms()

        try:
            # La nuova firma della funzione richiede il word_data completo
            risultato_ensemble = analyze_and_normalize_with_llm(
                transcript_whisper_rag          = testo_finale,
                transcrpit_whisper_testo_sporco = testo_sporco,
                transcript_PhoneticXeus         = ipa_transcript,
                word_data                       = word_data,
            )
        except Exception as e:
            print(f"Errore Ensemble LLM: {e}")
            risultato_ensemble = {"normalized_text": testo_finale, "semantic_issues": [], "detected_domain": "sconosciuto"}

        t_e5 = _now_ms()

        db["Pipeline_Stages"].update_one(
            {"_id": session_id},
            {
                "$set": {
                    # Aggiornato con la nuova label per il DB
                    "stages.4_llm_semantic_analysis": {"data": {"syntactic_analysis_enabled": False}, "started_at": t_s4, "saved_at": t_e5},
                    "stages.5_ensemble_llm": {"data": risultato_ensemble, "started_at": t_s4, "saved_at": t_e5}
                }
            }
        )

        # ════════════════════════════════════════════════════════
        # STEP 5.1 — PSEUDO-PERPLEXITY DI SESSIONE
        # ════════════════════════════════════════════════════════
        t_s51 = _now_ms()
        testo_normalizzato_it = risultato_ensemble.get("normalized_text", testo_finale)
        try:
            # Aggiornato con i nuovi parametri pppl_min e max
            pppl_session_result = compute_pppl(testo_normalizzato_it, pppl_min=1.98, pppl_max=30)
        except Exception as e:
            print(f"Errore PPPL: {e}")
            pppl_session_result = {"pppl": 0.0, "log_pppl": 0.0, "n_tokens": 0, "model": "Sconosciuto"}
        t_e51 = _now_ms()

        db["Pipeline_Stages"].update_one(
            {"_id": session_id},
            {"$set": {"stages.5_1_pppl_session": {"data": pppl_session_result, "started_at": t_s51, "saved_at": t_e51}}}
        )

        # ════════════════════════════════════════════════════════
        # STEP 6 — CLAIM SEGMENTATION (spaCy)
        # ════════════════════════════════════════════════════════
        t_s6 = _now_ms()
        try:
            claim_list = extract_spans_from_spacy(testo_normalizzato_it)
        except Exception as e:
            print(f"Errore Estrazione Claim: {e}")
            claim_list = [{"claim_text": testo_normalizzato_it, "start_token_i": 0, "end_token_i": 0, "n_tokens": 1}]
        t_e6 = _now_ms()

        db["Pipeline_Stages"].update_one(
            {"_id": session_id},
            {"$set": {"stages.6_claim_extraction": {"data": claim_list, "started_at": t_s6, "saved_at": t_e6}}}
        )

        # ════════════════════════════════════════════════════════
        # STEP 7 — CLAIM VALIDATION (LLM Semantic Alignment)
        # ════════════════════════════════════════════════════════
        t_s7 = _now_ms()
        try:
            # Usa word_data diretto e non più enriched_words
            validated_claims = validate_claims_with_llm(
                claim_list=claim_list,
                normalized_text=testo_normalizzato_it,
                word_data=word_data,
            )
        except Exception as e:
            print(f"Errore Validazione Claim: {e}")
            validated_claims = claim_list  # Fallback
        t_e7 = _now_ms()

        # ════════════════════════════════════════════════════════
        # STEP 7.1 — PPPL PER-CLAIM
        # ════════════════════════════════════════════════════════
        t_s71 = _now_ms()
        for vc in validated_claims:
            # Calcolo PPPL ora si fa su claim_text
            text_for_pppl = vc.get("claim_text", "")
            try:
                pppl_c = compute_pppl(text_for_pppl, pppl_min=1.66, pppl_max=160)
                vc["pppl_claim"]        = pppl_c.get("pppl")
                vc["pppl_claim_log"]    = pppl_c.get("log_pppl")
                vc["pppl_claim_tokens"] = pppl_c.get("n_tokens")
            except Exception:
                vc["pppl_claim"] = None
        t_e71 = _now_ms()

        # ════════════════════════════════════════════════════════
        # STEP 8 — RISK SCORING (Nuovo modello a due aliquote)
        # ════════════════════════════════════════════════════════
        t_s8 = _now_ms()
        try:
            scored_claims = score_all_claims(validated_claims)
            overall       = compute_overall_risk(scored_claims)
        except Exception as e:
            print(f"Errore calcolo Risk Score: {e}")
            scored_claims = validated_claims
            overall = {"overall_risk_level": "green", "overall_reason": "Errore nel calcolo del rischio."}
        t_e8 = _now_ms()

        db["Risk_Logs"].replace_one(
            {"_id": session_id},
            {
                "_id": session_id,
                "started_at": t_s8,
                "saved_at": t_e8,
                "timing": {
                    "claim_validation": {"started_at": t_s7, "saved_at": t_e7},
                    "pppl_claims":      {"started_at": t_s71, "saved_at": t_e71}
                },
                "overall": overall,
                "scored_claims": scored_claims
            },
            upsert=True
        )

        # ════════════════════════════════════════════════════════
        # STEP 9 — TRANSLATION METRICS
        # ════════════════════════════════════════════════════════
        t_s9 = _now_ms()
        translation_metrics = {
            "translated_text": testo_normalizzato_it,
            "prompt_text": None,
            "cosine_similarity": None,
            "wer": None,
            "bleu": None,
            "rouge1": None,
            "rouge2": None,
            "rougeL": None,
            "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2"
        }
        t_e9 = _now_ms()

        db["Translation_Metrics"].replace_one(
            {"_id": session_id},
            {
                "_id": session_id,
                "started_at": t_s9,
                "saved_at": t_e9,
                **translation_metrics
            },
            upsert=True
        )

        # Chiudiamo la sessione con successo usando la funzione di utility locale protetta
        _local_complete_session(session_id, success=True, overall_risk=overall)
        print(f"✅ Completata elaborazione realtime: {filename} ({overall['overall_risk_level']})")
        return {"status": "success", "session_id": session_id}

    except Exception as e:
        import traceback
        print("❌ CRASH PIPELINE REALTIME:")
        traceback.print_exc()
        _local_complete_session(session_id, success=False, error_msg=str(e))
        return {"status": "failed", "error": str(e)}

In [ ]:
## Nuova Cella - Tunnel Ngrok per l'esposizione esterna dell'API
!pip install -q pyngrok

from pyngrok import ngrok

# Inserisci il tuo Token Personale di Ngrok (creabile gratuitamente sul sito di ngrok)
NGROK_AUTH_TOKEN = "3EDkqkuQwAiMoim4f9zcuLrML8y_7uSQ1KazSS1aDxL3GZY4r"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Chiudi eventuali tunnel rimasti aperti in precedenza
ngrok.kill()

# Apri un tunnel HTTP sulla porta 8000
public_url = ngrok.connect(8000)
print("="*65)
print("🔗 ENDPOINT DA COPIARE DENTRO LA PAGINA 4 DI STREAMLIT:")
print(f"👉 {public_url.public_url}/process_audio 👈")
print("="*65)

# Avvia il server FastAPI in un thread separato in background per non bloccare il notebook
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info", use_colors=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 Server FastAPI avviato in background e pronto a ricevere file audio!")

In [ ]:
# Cella di emergenza per liberare la porta 8000
!fuser -k 8000/tcp
print("✅ Porta 8000 liberata con successo!")